# Phase 1 – Project environment setup

In [1]:
# Cell 1: Verify Python environment

import sys
import os

print("Python Version:", sys.version)
print("Python Executable:", sys.executable)
print("Project Working Directory:", os.getcwd())

Python Version: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
Python Executable: c:\Users\ADMIN\anaconda3\python.exe
Project Working Directory: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\python


In [2]:
# Cell 2: Check installed project dependencies

import importlib

required_libraries = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "sqlalchemy",
    "mysql.connector",
    "dotenv",
    "openpyxl"
]

for library in required_libraries:
    try:
        importlib.import_module(library)
        print(f"{library:<20} : Installed")
    except ImportError:
        print(f"{library:<20} : Not Installed")

pandas               : Installed
numpy                : Installed
matplotlib           : Installed
seaborn              : Installed
sqlalchemy           : Installed
mysql.connector      : Installed
dotenv               : Installed
openpyxl             : Installed


In [3]:
# Cell 3: Test Python–MySQL connectivity

import mysql.connector
from getpass import getpass

db_password = getpass("Enter MySQL root password: ")

try:
    connection = mysql.connector.connect(
        host="localhost",
        port=3306,
        user="root",
        password=db_password,
        connection_timeout=10
    )

    if connection.is_connected():
        cursor = connection.cursor()
        cursor.execute("SELECT VERSION()")
        mysql_version = cursor.fetchone()[0]

        print("MySQL Connection: SUCCESS")
        print("MySQL Version:", mysql_version)

        cursor.close()

except mysql.connector.Error as error:
    print("MySQL Connection: FAILED")
    print("Error:", error)

finally:
    if "connection" in locals() and connection.is_connected():
        connection.close()
        print("Connection closed successfully.")

MySQL Connection: SUCCESS
MySQL Version: 8.0.46
Connection closed successfully.


In [4]:
# Cell 4: Create Medallion database and schemas

import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password
)

cursor = connection.cursor()

cursor.execute("""
    CREATE DATABASE IF NOT EXISTS ecommerce_analytics
""")

cursor.execute("""
    USE ecommerce_analytics
""")

schemas = ["bronze", "silver", "gold"]

for schema in schemas:
    cursor.execute(f"""
        CREATE SCHEMA IF NOT EXISTS {schema}
    """)
    print(f"Schema created/verified: {schema}")

connection.commit()

cursor.close()
connection.close()

print("\nMedallion database setup completed.")

Schema created/verified: bronze
Schema created/verified: silver
Schema created/verified: gold

Medallion database setup completed.


In [5]:
# Cell 5: Verify database and schemas

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    SELECT SCHEMA_NAME
    FROM INFORMATION_SCHEMA.SCHEMATA
    WHERE SCHEMA_NAME IN ('bronze', 'silver', 'gold')
    ORDER BY SCHEMA_NAME
""")

for row in cursor.fetchall():
    print(row[0])

cursor.close()
connection.close()

bronze
gold
silver


In [7]:
# Cell 6: Configure project paths

from pathlib import Path

# Project root directory
PROJECT_ROOT = Path.cwd().parent

# Source dataset
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ecommerce_customer.csv"

print("Project Root:", PROJECT_ROOT)
print("Dataset Path:", DATA_PATH)
print("Dataset Exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print("Dataset Size:", round(DATA_PATH.stat().st_size / 1024, 2), "KB")
else:
    print("ERROR: Dataset not found. Check the file location.")

Project Root: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project
Dataset Path: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\data\raw\ecommerce_customer.csv
Dataset Exists: True
Dataset Size: 601.7 KB


In [8]:
# Cell 8: Load and inspect the source dataset

import pandas as pd

df_raw = pd.read_csv(DATA_PATH)

print("Dataset successfully loaded!")
print("-" * 45)

print("Number of Rows:", df_raw.shape[0])
print("Number of Columns:", df_raw.shape[1])

print("\nColumn Names:")
print(df_raw.columns.tolist())

print("\nFirst 5 Records:")
display(df_raw.head())

Dataset successfully loaded!
---------------------------------------------
Number of Rows: 3055
Number of Columns: 32

Column Names:
['Order_ID', 'Customer_ID', 'Order_Date', 'Customer_Age', 'Customer_Segment', 'City', 'Region', 'Category', 'Product_Name', 'Sales_Channel', 'Payment_Method', 'Device', 'Quantity', 'Unit_Price', 'Discount_Pct', 'Shipping_Days', 'Customer_Rating', 'Delivery_Rating', 'Return_Status', 'Order_Status', 'Marketing_Source', 'Customer_Loyalty_Points', 'Website_Session_Minutes', 'Items_Viewed', 'Previous_Orders', 'Customer_Satisfaction', 'Gross_Amount', 'Discount_Amount', 'Net_Sales', 'Estimated_Cost', 'Profit', 'Profit_Margin_Pct']

First 5 Records:


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Website_Session_Minutes,Items_Viewed,Previous_Orders,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
0,ORD100994,C1229,2025-06-07,44.0,Occasional,Delhi,Central,Sports,Shoes,Online,...,16.4,10,10,1.8,723.34,0.00,723.34,634.67,88.67,12.26
1,ORD101072,C1627,2025-03-18,51.0,Occasional,Kochi,West,Home Appliances,Shoes,Online,...,23.2,9,3,5.0,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD100995,C1410,2025-07-13,NaN,Occasional,Hyderabad,West,Electronics,Laptop,Online,...,9.1,10,6,3.0,739.05,105.68,633.37,370.35,263.02,41.53
3,ORD100731,C1578,2025-09-24,38.0,Regular,Bengaluru,North,Fashion,Shoes,Mobile App,...,22.7,4,4,3.6,28004.44,1473.03,26531.41,17911.62,8619.79,32.49
4,ORD102954,C1524,2025-06-09,39.0,Premium,Kolkata,South,Electronics,Skincare,Marketplace,...,6.6,8,8,2.5,9841.78,1842.38,7999.40,6214.63,1784.77,22.31


In [9]:
# Cell 9: Verify dataset data types and memory usage

print("Dataset Information:")
df_raw.info()

print("\nDataset Memory Usage:")
print(round(df_raw.memory_usage(deep=True).sum() / 1024, 2), "KB")

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 3055 entries, 0 to 3054
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Order_ID                 3055 non-null   str    
 1   Customer_ID              3055 non-null   str    
 2   Order_Date               3055 non-null   str    
 3   Customer_Age             2932 non-null   float64
 4   Customer_Segment         3055 non-null   str    
 5   City                     2979 non-null   str    
 6   Region                   3055 non-null   str    
 7   Category                 3055 non-null   str    
 8   Product_Name             3055 non-null   str    
 9   Sales_Channel            3055 non-null   str    
 10  Payment_Method           2978 non-null   str    
 11  Device                   3055 non-null   str    
 12  Quantity                 3055 non-null   int64  
 13  Unit_Price               2962 non-null   float64
 14  Discount_Pct  

In [10]:
# Cell 10: Complete source dataset profile

source_profile = pd.DataFrame({
    "Column": df_raw.columns,
    "Data_Type": df_raw.dtypes.astype(str).values,
    "Non_Null_Count": df_raw.notna().sum().values,
    "Missing_Count": df_raw.isna().sum().values,
    "Missing_Percentage": (
        df_raw.isna().mean().values * 100
    ).round(2)
})

display(source_profile)

print("\nTotal Records:", len(df_raw))
print("Total Columns:", len(df_raw.columns))
print("Total Missing Values:", df_raw.isna().sum().sum())

,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_Percentage
0,Order_ID,str,3055,0,0.00
1,Customer_ID,str,3055,0,0.00
2,Order_Date,str,3055,0,0.00
3,Customer_Age,float64,2932,123,4.03
4,Customer_Segment,str,3055,0,0.00
5,City,str,2979,76,2.49
6,Region,str,3055,0,0.00
7,Category,str,3055,0,0.00
8,Product_Name,str,3055,0,0.00
9,Sales_Channel,str,3055,0,0.00



Total Records: 3055
Total Columns: 32
Total Missing Values: 2518


In [11]:
# Cell 11: Save source data quality baseline

REPORT_DIR = PROJECT_ROOT / "reports" / "data_quality"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_PATH = REPORT_DIR / "source_data_profile.csv"

source_profile.to_csv(PROFILE_PATH, index=False)

print("Source profiling report saved successfully!")
print("Report Location:", PROFILE_PATH)

Source profiling report saved successfully!
Report Location: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\reports\data_quality\source_data_profile.csv


# Phase 2 – Bronze Layer: Raw Data Ingestion

In [12]:
# Cell 12: Create Bronze raw table

import mysql.connector

create_bronze_table = """
CREATE TABLE IF NOT EXISTS bronze.ecommerce_raw (

    Order_ID TEXT,
    Customer_ID TEXT,
    Order_Date TEXT,
    Customer_Age TEXT,
    Customer_Segment TEXT,
    City TEXT,
    Region TEXT,
    Category TEXT,
    Product_Name TEXT,
    Sales_Channel TEXT,
    Payment_Method TEXT,
    Device TEXT,
    Quantity TEXT,
    Unit_Price TEXT,
    Discount_Pct TEXT,
    Shipping_Days TEXT,
    Customer_Rating TEXT,
    Delivery_Rating TEXT,
    Return_Status TEXT,
    Order_Status TEXT,
    Marketing_Source TEXT,
    Customer_Loyalty_Points TEXT,
    Website_Session_Minutes TEXT,
    Items_Viewed TEXT,
    Previous_Orders TEXT,
    Customer_Satisfaction TEXT,
    Gross_Amount TEXT,
    Discount_Amount TEXT,
    Net_Sales TEXT,
    Estimated_Cost TEXT,
    Profit TEXT,
    Profit_Margin_Pct TEXT,

    ingestion_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    source_file VARCHAR(255),
    ingestion_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP

);
"""

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()
cursor.execute(create_bronze_table)

print("Bronze raw table created or already exists.")

cursor.close()
connection.close()

Bronze raw table created or already exists.


In [13]:
# Cell 13: Verify Bronze table columns

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    DESCRIBE bronze.ecommerce_raw
""")

for column in cursor.fetchall():
    print(column[0], "|", column[1], "|", column[3])

cursor.close()
connection.close()

Order_ID | text | 
Customer_ID | text | 
Order_Date | text | 
Customer_Age | text | 
Customer_Segment | text | 
City | text | 
Region | text | 
Category | text | 
Product_Name | text | 
Sales_Channel | text | 
Payment_Method | text | 
Device | text | 
Quantity | text | 
Unit_Price | text | 
Discount_Pct | text | 
Shipping_Days | text | 
Customer_Rating | text | 
Delivery_Rating | text | 
Return_Status | text | 
Order_Status | text | 
Marketing_Source | text | 
Customer_Loyalty_Points | text | 
Website_Session_Minutes | text | 
Items_Viewed | text | 
Previous_Orders | text | 
Customer_Satisfaction | text | 
Gross_Amount | text | 
Discount_Amount | text | 
Net_Sales | text | 
Estimated_Cost | text | 
Profit | text | 
Profit_Margin_Pct | text | 
ingestion_id | bigint | PRI
source_file | varchar(255) | 
ingestion_timestamp | datetime | 


In [14]:
# Cell 14: Validate Bronze structure and prepare raw CSV

import pandas as pd
import mysql.connector

# Reload source without automatic type conversion or NA interpretation
df_bronze = pd.read_csv(
    DATA_PATH,
    dtype=str,
    keep_default_na=False
)

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

# Retrieve Bronze table structure
cursor.execute("DESCRIBE bronze.ecommerce_raw")
bronze_columns = [row[0] for row in cursor.fetchall()]

# Check existing records
cursor.execute("SELECT COUNT(*) FROM bronze.ecommerce_raw")
existing_records = cursor.fetchone()[0]

cursor.close()
connection.close()

print("Source columns:", len(df_bronze.columns))
print("Bronze table columns:", len(bronze_columns))
print("Bronze metadata columns:", len(bronze_columns) - len(df_bronze.columns))

print("\nSource and Bronze column names match:",
      list(df_bronze.columns) == bronze_columns[:32])

print("Source records:", len(df_bronze))
print("Existing Bronze records:", existing_records)

print("\nRaw data types:")
print(df_bronze.dtypes.value_counts())

print("\nRaw CSV preview:")
display(df_bronze.head(3))

Source columns: 32
Bronze table columns: 35
Bronze metadata columns: 3

Source and Bronze column names match: True
Source records: 3055
Existing Bronze records: 0

Raw data types:
str    32
Name: count, dtype: int64

Raw CSV preview:


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Website_Session_Minutes,Items_Viewed,Previous_Orders,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
0,ORD100994,C1229,2025-06-07,44.0,Occasional,Delhi,Central,Sports,Shoes,Online,...,16.4,10,10,1.8,723.34,0.0,723.34,634.67,88.67,12.26
1,ORD101072,C1627,2025-03-18,51.0,Occasional,Kochi,West,Home Appliances,Shoes,Online,...,23.2,9,3,5.0,,,,,,
2,ORD100995,C1410,2025-07-13,,Occasional,Hyderabad,West,Electronics,Laptop,Online,...,9.1,10,6,3.0,739.05,105.68,633.37,370.35,263.02,41.53


In [15]:
# Cell 15: Bronze layer - Raw data ingestion

import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

try:
    # Prevent accidental duplicate ingestion
    cursor.execute("SELECT COUNT(*) FROM bronze.ecommerce_raw")
    existing_records = cursor.fetchone()[0]

    if existing_records > 0:
        raise Exception(
            f"Bronze table already contains {existing_records} records. "
            "Ingestion cancelled."
        )

    # Prepare source columns
    source_columns = list(df_bronze.columns)

    insert_columns = source_columns + ["source_file"]

    column_names = ", ".join(
        f"`{column}`" for column in insert_columns
    )

    placeholders = ", ".join(["%s"] * len(insert_columns))

    insert_query = f"""
        INSERT INTO bronze.ecommerce_raw ({column_names})
        VALUES ({placeholders})
    """

    # Prepare records with ingestion metadata
    records = [
        tuple(row) + (DATA_PATH.name,)
        for row in df_bronze.itertuples(index=False, name=None)
    ]

    # Batch insertion
    batch_size = 500

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        cursor.executemany(insert_query, batch)

        print(
            f"Inserted batch: "
            f"{start + 1} - {start + len(batch)}"
        )

    connection.commit()

    print("\nBronze ingestion completed successfully!")
    print("Total records inserted:", cursor.rowcount)

except Exception as error:
    connection.rollback()
    print("\nIngestion failed. Transaction rolled back.")
    print("Error:", error)

finally:
    cursor.close()
    connection.close()

Inserted batch: 1 - 500
Inserted batch: 501 - 1000
Inserted batch: 1001 - 1500
Inserted batch: 1501 - 2000
Inserted batch: 2001 - 2500
Inserted batch: 2501 - 3000
Inserted batch: 3001 - 3055

Bronze ingestion completed successfully!
Total records inserted: 55


In [16]:
# Cell 16: Verify Bronze ingestion

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

# Total records in Bronze
cursor.execute("""
    SELECT COUNT(*)
    FROM bronze.ecommerce_raw
""")

bronze_count = cursor.fetchone()[0]

# Verify source file metadata
cursor.execute("""
    SELECT
        source_file,
        COUNT(*) AS record_count,
        COUNT(DISTINCT ingestion_id) AS unique_ingestion_ids
    FROM bronze.ecommerce_raw
    GROUP BY source_file
""")

metadata_result = cursor.fetchall()

print("Source Records:", len(df_bronze))
print("Bronze Records:", bronze_count)
print("Record Count Match:", len(df_bronze) == bronze_count)

print("\nIngestion Metadata:")
for row in metadata_result:
    print(row)

cursor.close()
connection.close()

Source Records: 3055
Bronze Records: 3055
Record Count Match: True

Ingestion Metadata:
('ecommerce_customer.csv', 3055, 3055)


In [17]:
# Cell 17: Exact source-to-Bronze comparison

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

source_columns = list(df_bronze.columns)

column_names = ", ".join(
    f"`{column}`" for column in source_columns
)

cursor.execute(f"""
    SELECT {column_names}
    FROM bronze.ecommerce_raw
    ORDER BY ingestion_id
""")

bronze_records = cursor.fetchall()

cursor.close()
connection.close()

# Reconstruct Bronze DataFrame
df_bronze_db = pd.DataFrame(
    bronze_records,
    columns=source_columns
)

# Compare source and Bronze values
comparison = df_bronze.reset_index(drop=True).eq(
    df_bronze_db.reset_index(drop=True)
)

# Count mismatches
mismatch_count = int((~comparison).sum().sum())

column_mismatches = (~comparison).sum()

print("Source Records:", len(df_bronze))
print("Bronze Records:", len(df_bronze_db))
print("Total Columns Compared:", len(source_columns))
print("Total Values Compared:", df_bronze.size)
print("Total Mismatched Values:", mismatch_count)

print("\nExact Data Match:", mismatch_count == 0)

print("\nColumn-wise Mismatch Summary:")
display(column_mismatches[column_mismatches > 0])

Source Records: 3055
Bronze Records: 3055
Total Columns Compared: 32
Total Values Compared: 97760
Total Mismatched Values: 0

Exact Data Match: True

Column-wise Mismatch Summary:


Series([], dtype: int64)

In [18]:
# Cell 18: Bronze layer audit report

bronze_audit = {
    "Source_File": DATA_PATH.name,
    "Source_Records": len(df_bronze),
    "Bronze_Records": len(df_bronze_db),
    "Source_Columns": len(df_bronze.columns),
    "Total_Values_Compared": df_bronze.size,
    "Mismatched_Values": mismatch_count,
    "Unique_Ingestion_IDs": bronze_count,
    "Record_Count_Match": len(df_bronze) == bronze_count,
    "Exact_Data_Match": mismatch_count == 0,
    "Ingestion_Status": "SUCCESS"
}

bronze_audit_df = pd.DataFrame(
    [bronze_audit]
)

BRONZE_AUDIT_PATH = (
    REPORT_DIR / "bronze_ingestion_audit.csv"
)

bronze_audit_df.to_csv(
    BRONZE_AUDIT_PATH,
    index=False
)

display(bronze_audit_df)

print("\nAudit report saved:", BRONZE_AUDIT_PATH)

,Source_File,Source_Records,Bronze_Records,Source_Columns,Total_Values_Compared,Mismatched_Values,Unique_Ingestion_IDs,Record_Count_Match,Exact_Data_Match,Ingestion_Status
0,ecommerce_customer.csv,3055,3055,32,97760,0,3055,True,True,SUCCESS



Audit report saved: c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\reports\data_quality\bronze_ingestion_audit.csv


In [19]:
# Cell 19: Load Bronze data for Silver processing

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

df_silver_source = pd.read_sql(
    "SELECT * FROM bronze.ecommerce_raw",
    connection
)

connection.close()

print("Bronze data loaded successfully!")
print("Rows:", df_silver_source.shape[0])
print("Columns:", df_silver_source.shape[1])

display(df_silver_source.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14864\4341804.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_silver_source = pd.read_sql(


Bronze data loaded successfully!
Rows: 3055
Columns: 35


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct,ingestion_id,source_file,ingestion_timestamp
0,ORD100994,C1229,2025-06-07,44.0,Occasional,Delhi,Central,Sports,Shoes,Online,...,1.8,723.34,0.0,723.34,634.67,88.67,12.26,1,ecommerce_customer.csv,2026-11-20 08:42:40
1,ORD101072,C1627,2025-03-18,51.0,Occasional,Kochi,West,Home Appliances,Shoes,Online,...,5.0,,,,,,,2,ecommerce_customer.csv,2026-11-20 08:42:40
2,ORD100995,C1410,2025-07-13,,Occasional,Hyderabad,West,Electronics,Laptop,Online,...,3.0,739.05,105.68,633.37,370.35,263.02,41.53,3,ecommerce_customer.csv,2026-11-20 08:42:40
3,ORD100731,C1578,2025-09-24,38.0,Regular,Bengaluru,North,Fashion,Shoes,Mobile App,...,3.6,28004.44,1473.03,26531.41,17911.62,8619.79,32.49,4,ecommerce_customer.csv,2026-11-20 08:42:40
4,ORD102954,C1524,2025-06-09,39.0,Premium,Kolkata,South,Electronics,Skincare,Marketplace,...,2.5,9841.78,1842.38,7999.4,6214.63,1784.77,22.31,5,ecommerce_customer.csv,2026-11-20 08:42:40


# Phase 3 – Silver Layer

In [20]:
# Cell 20: Bronze data quality assessment

from sqlalchemy import create_engine, URL

# Create SQLAlchemy connection
db_url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password=db_password,
    host="localhost",
    port=3306,
    database="ecommerce_analytics"
)

engine = create_engine(db_url)

# Load persisted Bronze data
df_silver_source = pd.read_sql_query(
    "SELECT * FROM bronze.ecommerce_raw",
    engine
)

# Separate business columns and metadata
business_columns = df_bronze.columns.tolist()

df_business = df_silver_source[business_columns].copy()

# Convert empty strings to missing values for profiling only
df_profile = df_business.replace(r"^\s*$", pd.NA, regex=True)

# Column-level quality profile
bronze_quality_profile = pd.DataFrame({
    "Column": business_columns,
    "Missing_Count": df_profile.isna().sum().values,
    "Missing_Percentage": (
        df_profile.isna().mean().values * 100
    ).round(2),
    "Unique_Values": df_profile.nunique().values
})

# Duplicate analysis
duplicate_order_count = (
    df_business["Order_ID"].duplicated(keep=False).sum()
)

duplicate_record_count = df_business.duplicated().sum()

print("BRONZE DATA QUALITY SUMMARY")
print("-" * 45)

print("Total Records:", len(df_business))
print("Total Columns:", len(df_business.columns))
print("Unique Orders:", df_business["Order_ID"].nunique())
print("Unique Customers:", df_business["Customer_ID"].nunique())

print("Duplicate Order_ID Records:", duplicate_order_count)
print("Duplicate Complete Records:", duplicate_record_count)

print("\nColumn-Level Missing Value Profile:")
display(bronze_quality_profile)

engine.dispose()

BRONZE DATA QUALITY SUMMARY
---------------------------------------------
Total Records: 3055
Total Columns: 32
Unique Orders: 3000
Unique Customers: 776
Duplicate Order_ID Records: 110
Duplicate Complete Records: 50

Column-Level Missing Value Profile:


,Column,Missing_Count,Missing_Percentage,Unique_Values
0,Order_ID,0,0.00,3000
1,Customer_ID,0,0.00,776
2,Order_Date,0,0.00,365
3,Customer_Age,123,4.03,74
4,Customer_Segment,0,0.00,4
5,City,75,2.45,21
6,Region,0,0.00,5
7,Category,0,0.00,16
8,Product_Name,0,0.00,10
9,Sales_Channel,0,0.00,4


In [21]:
# Cell 21: Identify duplicate orders

# Find all records belonging to duplicated Order_IDs
duplicate_orders = df_business[
    df_business["Order_ID"].duplicated(keep=False)
].copy()

# Sort for easier investigation
duplicate_orders = duplicate_orders.sort_values(
    by=["Order_ID", "Customer_ID"]
)

# Count records per Order_ID
duplicate_summary = (
    duplicate_orders.groupby("Order_ID")
    .agg(
        Record_Count=("Order_ID", "size"),
        Unique_Customers=("Customer_ID", "nunique"),
        Unique_Order_Status=("Order_Status", "nunique")
    )
    .reset_index()
)

print("Duplicate Order IDs:", duplicate_summary.shape[0])
print("Records involved:", len(duplicate_orders))

print("\nDuplicate Order Summary:")
display(duplicate_summary.head(20))

print("\nDetailed Duplicate Records:")
display(duplicate_orders.head(20))

Duplicate Order IDs: 55
Records involved: 110

Duplicate Order Summary:


,Order_ID,Record_Count,Unique_Customers,Unique_Order_Status
0,ORD100045,2,1,1
1,ORD100053,2,1,1
2,ORD100252,2,1,1
3,ORD100260,2,1,1
4,ORD100290,2,1,1
5,ORD100292,2,1,1
6,ORD100297,2,1,1
7,ORD100341,2,1,1
8,ORD100409,2,1,1
9,ORD100427,2,1,2



Detailed Duplicate Records:


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Website_Session_Minutes,Items_Viewed,Previous_Orders,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
888,ORD100045,C1152,2025-01-25,34.0,Premium,Bengaluru,South,Home Appliances,Shoes,Online,...,6.0,4,5,3.2,2371.46,29.88,2341.58,1517.05,824.53,35.21
1514,ORD100045,C1152,2025-01-25,34.0,Premium,Bengaluru,South,Home Appliances,Shoes,Online,...,6.0,4,5,3.2,2371.46,29.88,2341.58,1517.05,824.53,35.21
1814,ORD100053,C1602,2025-05-13,30.0,Regular,Bengaluru,North,Furniture,Treadmill,Online,...,21.1,7,4,4.5,18455.28,3532.34,14922.94,8850.48,6072.46,40.69
1964,ORD100053,C1602,2025-05-13,30.0,Regular,Bengaluru,North,Furniture,Treadmill,Online,...,21.1,7,4,4.5,18455.28,3532.34,14922.94,8850.48,6072.46,40.69
1276,ORD100252,C1598,2025-06-30,43.0,Regular,Kolkata,West,Books,Shirt,Online,...,16.0,7,6,4.3,6712.56,36.25,6676.31,5004.13,1672.18,25.05
1678,ORD100252,C1598,2025-06-30,43.0,Regular,Kolkata,West,Books,Shirt,Online,...,16.0,7,6,4.3,6712.56,36.25,6676.31,5004.13,1672.18,25.05
1118,ORD100260,C1329,2025-08-03,36.0,Regular,,West,Sports,Chair,Online,...,19.6,11,8,4.7,2792.16,800.79,1991.37,1646.86,344.51,17.3
2863,ORD100260,C1329,2025-08-03,36.0,Regular,,West,Sports,Chair,Online,...,19.6,11,8,4.7,2792.16,800.79,1991.37,1646.86,344.51,17.3
31,ORD100290,C1149,2025-06-22,17.0,Regular,Kolkata,South,Sports,Skincare,Mobile App,...,11.1,12,5,4.6,2540.22,7.87,2532.35,1967.18,565.17,22.32
2950,ORD100290,C1149,2025-06-22,17.0,Regular,Kolkata,South,Sports,Skincare,Mobile App,...,11.1,12,5,4.6,2540.22,7.87,2532.35,1967.18,565.17,22.32


In [24]:
# Cell 22: Classify exact and conflicting duplicate orders

# Count total records per Order_ID
record_counts = (
    duplicate_orders
    .groupby("Order_ID")
    .size()
    .rename("Record_Count")
)

# Remove identical rows temporarily to identify distinct records
distinct_records = duplicate_orders[
    business_columns
].drop_duplicates()

# Count distinct records per Order_ID
unique_counts = (
    distinct_records
    .groupby("Order_ID")
    .size()
    .rename("Unique_Records")
)

# Combine both counts
duplicate_classification = pd.concat(
    [record_counts, unique_counts],
    axis=1
).reset_index()

# Classify duplicate groups
duplicate_classification["Duplicate_Type"] = (
    duplicate_classification["Unique_Records"]
    .map({
        1: "Exact Duplicate"
    })
    .fillna("Conflicting Duplicate")
)

print("Duplicate Classification Summary:")

display(
    duplicate_classification["Duplicate_Type"]
    .value_counts()
    .rename_axis("Duplicate_Type")
    .reset_index(name="Group_Count")
)

print("\nConflicting Duplicate Orders:")

display(
    duplicate_classification[
        duplicate_classification["Duplicate_Type"]
        == "Conflicting Duplicate"
    ]
)

Duplicate Classification Summary:


,Duplicate_Type,Group_Count
0,Exact Duplicate,50
1,Conflicting Duplicate,5



Conflicting Duplicate Orders:


,Order_ID,Record_Count,Unique_Records,Duplicate_Type
9,ORD100427,2,2,Conflicting Duplicate
13,ORD100700,2,2,Conflicting Duplicate
33,ORD101794,2,2,Conflicting Duplicate
38,ORD102113,2,2,Conflicting Duplicate
40,ORD102116,2,2,Conflicting Duplicate


In [25]:
# Cell 23: Find columns containing conflicting values

conflicting_orders = duplicate_classification.loc[
    duplicate_classification["Duplicate_Type"]
    == "Conflicting Duplicate",
    "Order_ID"
]

conflict_details = []

for order_id in conflicting_orders:

    order_records = duplicate_orders[
        duplicate_orders["Order_ID"] == order_id
    ]

    for column in business_columns:

        unique_values = order_records[column].nunique(
            dropna=False
        )

        if unique_values > 1:

            conflict_details.append({
                "Order_ID": order_id,
                "Column": column,
                "Unique_Values": unique_values,
                "Values": order_records[column].tolist()
            })

conflict_details_df = pd.DataFrame(conflict_details)

display(conflict_details_df)

,Order_ID,Column,Unique_Values,Values
0,ORD100427,Order_Status,2,"[Pending, Delivered]"
1,ORD100700,Order_Status,2,"[Delayed, Delivered]"
2,ORD101794,Order_Status,2,"[Pending, Delivered]"
3,ORD102113,Order_Status,2,"[Returned, Delivered]"
4,ORD102116,Order_Status,2,"[Delivered, Delayed]"


In [26]:
# Cell 24: Verify conflicting records differ only by Order_Status

conflict_columns = []

for order_id in conflicting_orders:

    group = duplicate_orders[
        duplicate_orders["Order_ID"] == order_id
    ]

    different_columns = [
        column
        for column in business_columns
        if group[column].nunique(dropna=False) > 1
    ]

    conflict_columns.append({
        "Order_ID": order_id,
        "Conflicting_Columns": different_columns,
        "Conflict_Count": len(different_columns)
    })

conflict_validation = pd.DataFrame(conflict_columns)

display(conflict_validation)

print(
    "\nAll conflicts limited to Order_Status:",
    conflict_validation["Conflicting_Columns"]
    .apply(lambda cols: cols == ["Order_Status"])
    .all()
)

,Order_ID,Conflicting_Columns,Conflict_Count
0,ORD100427,[Order_Status],1
1,ORD100700,[Order_Status],1
2,ORD101794,[Order_Status],1
3,ORD102113,[Order_Status],1
4,ORD102116,[Order_Status],1



All conflicts limited to Order_Status: True


In [27]:
# Cell 25: Verify conflicting orders directly in MySQL

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

conflict_query = """
SELECT
    Order_ID,
    COUNT(*) AS Record_Count,
    COUNT(DISTINCT Order_Status) AS Status_Count,
    GROUP_CONCAT(
        DISTINCT Order_Status
        ORDER BY Order_Status
        SEPARATOR ' / '
    ) AS Status_Values
FROM bronze.ecommerce_raw
WHERE Order_ID IN (
    'ORD100427',
    'ORD100700',
    'ORD101794',
    'ORD102113',
    'ORD102116'
)
GROUP BY Order_ID
ORDER BY Order_ID;
"""

cursor.execute(conflict_query)

sql_conflict_results = cursor.fetchall()

for row in sql_conflict_results:
    print(row)

cursor.close()
connection.close()

('ORD100427', 2, 2, 'Delivered / Pending')
('ORD100700', 2, 2, 'Delayed / Delivered')
('ORD101794', 2, 2, 'Delivered / Pending')
('ORD102113', 2, 2, 'Delivered / Returned')
('ORD102116', 2, 2, 'Delayed / Delivered')


In [28]:
# Cell 26: Create Silver duplicate exceptions table

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS silver.ecommerce_duplicate_exceptions
    LIKE bronze.ecommerce_raw
""")

print("Silver duplicate exceptions table created or verified.")

cursor.close()
connection.close()

Silver duplicate exceptions table created or verified.


In [29]:
# Cell 27: Create Silver deduplicated candidate table

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS silver.ecommerce_deduplicated
    LIKE bronze.ecommerce_raw
""")

print("Silver deduplicated table created or verified.")

cursor.close()
connection.close()

Silver deduplicated table created or verified.


In [30]:
# Cell 28: Move conflicting orders into Silver exceptions

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

try:
    # Check whether the target table is empty
    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.ecommerce_duplicate_exceptions
    """)

    existing_count = cursor.fetchone()[0]

    if existing_count > 0:
        raise Exception(
            "Exceptions table is not empty. "
            "Insertion cancelled to prevent duplicates."
        )

    # Preserve all original columns and ingestion metadata
    all_columns = business_columns + [
        "ingestion_id",
        "source_file",
        "ingestion_timestamp"
    ]

    column_list = ", ".join(
        f"`{column}`" for column in all_columns
    )

    insert_query = f"""
        INSERT INTO silver.ecommerce_duplicate_exceptions
        ({column_list})
        SELECT {column_list}
        FROM bronze.ecommerce_raw
        WHERE Order_ID IN (
            'ORD100427',
            'ORD100700',
            'ORD101794',
            'ORD102113',
            'ORD102116'
        )
    """

    cursor.execute(insert_query)
    connection.commit()

    print("Conflicting records preserved successfully!")
    print("Records inserted:", cursor.rowcount)

except Exception as error:
    connection.rollback()
    print("Insertion failed:", error)

finally:
    cursor.close()
    connection.close()

Conflicting records preserved successfully!
Records inserted: 10


In [31]:
# Cell 29: Deduplicate records using SQL ROW_NUMBER()

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

try:
    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.ecommerce_deduplicated
    """)

    existing_count = cursor.fetchone()[0]

    if existing_count > 0:
        raise Exception(
            "Silver table already contains records. "
            "Insertion cancelled."
        )

    columns = business_columns + [
        "ingestion_id",
        "source_file",
        "ingestion_timestamp"
    ]

    column_list = ", ".join(
        f"`{column}`" for column in columns
    )

    deduplication_query = f"""
        INSERT INTO silver.ecommerce_deduplicated ({column_list})

        SELECT {column_list}
        FROM (
            SELECT
                b.*,

                ROW_NUMBER() OVER (
                    PARTITION BY
                        {", ".join(f"`{col}`" for col in business_columns)}
                    ORDER BY ingestion_id
                ) AS row_num

            FROM bronze.ecommerce_raw b

            WHERE NOT EXISTS (
                SELECT 1
                FROM silver.ecommerce_duplicate_exceptions e
                WHERE e.ingestion_id = b.ingestion_id
            )

        ) AS ranked_records

        WHERE row_num = 1
    """

    cursor.execute(deduplication_query)
    connection.commit()

    print("Silver deduplication completed!")
    print("Records inserted:", cursor.rowcount)

except Exception as error:
    connection.rollback()
    print("Deduplication failed:", error)

finally:
    cursor.close()
    connection.close()

Silver deduplication completed!
Records inserted: 2995


In [32]:
# Cell 30: Validate Bronze and Silver record counts

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

validation_query = """
SELECT 'Bronze' AS Layer, COUNT(*) AS Records
FROM bronze.ecommerce_raw

UNION ALL

SELECT 'Silver Deduplicated', COUNT(*)
FROM silver.ecommerce_deduplicated

UNION ALL

SELECT 'Silver Exceptions', COUNT(*)
FROM silver.ecommerce_duplicate_exceptions;
"""

cursor.execute(validation_query)

results = cursor.fetchall()

for row in results:
    print(f"{row[0]}: {row[1]}")

cursor.close()
connection.close()

Bronze: 3055
Silver Deduplicated: 2995
Silver Exceptions: 10


In [33]:
# Cell 31: Verify duplicate Order_IDs in Silver candidate

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

cursor = connection.cursor()

cursor.execute("""
    SELECT
        Order_ID,
        COUNT(*) AS Record_Count
    FROM silver.ecommerce_deduplicated
    GROUP BY Order_ID
    HAVING COUNT(*) > 1
""")

remaining_duplicates = cursor.fetchall()

print("Remaining duplicate Order_ID groups:",
      len(remaining_duplicates))

if remaining_duplicates:
    print(remaining_duplicates)

cursor.close()
connection.close()

Remaining duplicate Order_ID groups: 0


In [34]:
# Cell 32: Analyze missing values in Silver candidate

connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=db_password,
    database="ecommerce_analytics"
)

df_silver_candidate = pd.read_sql_query(
    """
    SELECT *
    FROM silver.ecommerce_deduplicated
    """,
    engine
)

connection.close()

# Business columns only
df_silver_business = df_silver_candidate[
    business_columns
].copy()

# Treat empty and whitespace-only strings as missing for analysis
df_silver_profile = df_silver_business.replace(
    r"^\s*$",
    pd.NA,
    regex=True
)

silver_missing_profile = pd.DataFrame({
    "Column": business_columns,
    "Missing_Count": df_silver_profile.isna().sum().values,
    "Missing_Percentage": (
        df_silver_profile.isna().mean().values * 100
    ).round(2)
})

silver_missing_profile = silver_missing_profile.sort_values(
    by="Missing_Count",
    ascending=False
)

print("Silver Candidate Records:", len(df_silver_business))
print(
    "Total Missing Cells:",
    df_silver_profile.isna().sum().sum()
)

display(silver_missing_profile)

Silver Candidate Records: 2995
Total Missing Cells: 2446


,Column,Missing_Count,Missing_Percentage
31,Profit_Margin_Pct,239,7.98
28,Net_Sales,238,7.95
29,Estimated_Cost,238,7.95
27,Discount_Amount,238,7.95
30,Profit,238,7.95
16,Customer_Rating,209,6.98
14,Discount_Pct,149,4.97
25,Customer_Satisfaction,149,4.97
3,Customer_Age,120,4.01
21,Customer_Loyalty_Points,119,3.97


In [35]:
# Cell 33: Analyze missing-value patterns in financial columns

financial_columns = [
    "Unit_Price",
    "Gross_Amount",
    "Discount_Pct",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

financial_data = df_silver_business[
    financial_columns
].replace(r"^\s*$", pd.NA, regex=True)

financial_missing = financial_data.isna()

print("Missing Values by Financial Column:")
display(financial_missing.sum().to_frame("Missing_Count"))

print("\nFinancial Missing-Value Pattern:")

pattern_summary = (
    financial_missing
    .astype(int)
    .groupby(financial_columns)
    .size()
    .reset_index(name="Record_Count")
    .sort_values("Record_Count", ascending=False)
)

display(pattern_summary)

print("\nRecords with All Financial Columns Missing:",
      financial_missing.all(axis=1).sum())

print("Records with At Least One Financial Value Missing:",
      financial_missing.any(axis=1).sum())

Missing Values by Financial Column:


,Missing_Count
Unit_Price,90
Gross_Amount,90
Discount_Pct,149
Discount_Amount,238
Net_Sales,238
Estimated_Cost,238
Profit,238
Profit_Margin_Pct,239



Financial Missing-Value Pattern:


,Unit_Price,Gross_Amount,Discount_Pct,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct,Record_Count
0,0,0,0,0,0,0,0,0,2756
2,0,0,1,1,1,1,1,1,148
3,1,1,0,1,1,1,1,1,89
1,0,0,0,0,0,0,0,1,1
4,1,1,1,1,1,1,1,1,1



Records with All Financial Columns Missing: 1
Records with At Least One Financial Value Missing: 239


In [36]:
# Cell 34: Detect hidden missing-value representations

missing_tokens = {
    "", "nan", "null", "none", "n/a", "na", "nat"
}

token_summary = []

for column in business_columns:

    normalized = (
        df_silver_business[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    token_count = normalized.isin(missing_tokens).sum()

    if token_count > 0:
        token_summary.append({
            "Column": column,
            "Missing_Token_Count": int(token_count)
        })

token_summary_df = pd.DataFrame(token_summary)

print("Missing-value token summary:")
display(token_summary_df)

Missing-value token summary:


,Column,Missing_Token_Count
0,Customer_Age,120
1,City,75
2,Payment_Method,75
3,Unit_Price,90
4,Discount_Pct,149
5,Shipping_Days,105
6,Customer_Rating,209
7,Customer_Loyalty_Points,119
8,Website_Session_Minutes,75
9,Customer_Satisfaction,149


In [37]:
# Cell 35: Inspect records with missing financial information

financial_missing = financial_data.isna()

affected_records = df_silver_business.loc[
    financial_missing.any(axis=1),
    [
        "Order_ID",
        "Product_Name",
        "Category",
        "Quantity",
        "Unit_Price",
        "Gross_Amount",
        "Discount_Pct",
        "Discount_Amount",
        "Net_Sales",
        "Estimated_Cost",
        "Profit",
        "Profit_Margin_Pct"
    ]
].copy()

print("Affected Records:", len(affected_records))

display(affected_records.head(15))

Affected Records: 239


,Order_ID,Product_Name,Category,Quantity,Unit_Price,Gross_Amount,Discount_Pct,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
1,ORD101072,Shoes,Home Appliances,2,,,28.01,,,,,
22,ORD101810,Chair,Home Appliances,1,5160.28,5160.28,,,,,,
28,ORD102765,Shoes,Furniture,3,,,25.52,,,,,
32,ORD100424,Mixer,Books,5,1602.68,8013.4,,,,,,
34,ORD100823,Headphones,Sports,4,1450.1,5800.4,,,,,,
38,ORD101124,Treadmill,Books,2,402.93,805.86,,,,,,
42,ORD102595,Treadmill,Books,4,1250.58,5002.32,,,,,,
45,ORD101424,Headphones,Books,2,324.54,649.08,,,,,,
52,ORD100026,Rice,Electronics,2,,,22.36,,,,,
71,ORD101361,Novel,Home Appliances,2,,,16.36,,,,,


In [38]:
# Cell 36: Validate financial relationships using complete records

financial_check = df_silver_business.copy()

numeric_columns = [
    "Quantity",
    "Unit_Price",
    "Gross_Amount",
    "Discount_Pct",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

for col in numeric_columns:
    financial_check[col] = pd.to_numeric(
        financial_check[col],
        errors="coerce"
    )

# Validate each formula only where its required inputs exist
checks = {
    "Gross = Quantity * Unit Price": [
        "Quantity", "Unit_Price", "Gross_Amount"
    ],
    "Discount = Gross * Discount %": [
        "Gross_Amount", "Discount_Pct", "Discount_Amount"
    ],
    "Net Sales = Gross - Discount": [
        "Gross_Amount", "Discount_Amount", "Net_Sales"
    ],
    "Profit = Net Sales - Cost": [
        "Net_Sales", "Estimated_Cost", "Profit"
    ],
    "Margin = Profit / Net Sales * 100": [
        "Profit", "Net_Sales", "Profit_Margin_Pct"
    ]
}

results = []

for formula, columns in checks.items():

    valid = financial_check[columns].notna().all(axis=1)
    data = financial_check.loc[valid]

    if formula == "Gross = Quantity * Unit Price":
        expected = data["Quantity"] * data["Unit_Price"]
        actual = data["Gross_Amount"]

    elif formula == "Discount = Gross * Discount %":
        expected = data["Gross_Amount"] * data["Discount_Pct"] / 100
        actual = data["Discount_Amount"]

    elif formula == "Net Sales = Gross - Discount":
        expected = data["Gross_Amount"] - data["Discount_Amount"]
        actual = data["Net_Sales"]

    elif formula == "Profit = Net Sales - Cost":
        expected = data["Net_Sales"] - data["Estimated_Cost"]
        actual = data["Profit"]

    else:
        expected = (
            data["Profit"] / data["Net_Sales"] * 100
        )
        actual = data["Profit_Margin_Pct"]

    difference = (expected - actual).abs()

    results.append({
        "Formula": formula,
        "Comparable_Records": len(data),
        "Matching_Records": int((difference <= 0.02).sum()),
        "Mismatching_Records": int((difference > 0.02).sum()),
        "Max_Absolute_Difference": round(
            difference.max(), 4
        ) if len(difference) else None
    })

formula_validation = pd.DataFrame(results)

display(formula_validation)

,Formula,Comparable_Records,Matching_Records,Mismatching_Records,Max_Absolute_Difference
0,Gross = Quantity * Unit Price,2905,2905,0,0.000
1,Discount = Gross * Discount %,2757,2757,0,0.005
2,Net Sales = Gross - Discount,2757,2757,0,0.000
3,Profit = Net Sales - Cost,2757,2757,0,0.000
4,Margin = Profit / Net Sales * 100,2756,2756,0,0.005


In [39]:
# Cell 37: Count records where financial values can be reconstructed

df_check = financial_check.copy()

reconstruction_checks = {
    "Gross_Amount": (
        df_check["Gross_Amount"].isna()
        & df_check["Quantity"].notna()
        & df_check["Unit_Price"].notna()
    ),

    "Discount_Amount": (
        df_check["Discount_Amount"].isna()
        & df_check["Gross_Amount"].notna()
        & df_check["Discount_Pct"].notna()
    ),

    "Net_Sales": (
        df_check["Net_Sales"].isna()
        & df_check["Gross_Amount"].notna()
        & df_check["Discount_Amount"].notna()
    ),

    "Profit": (
        df_check["Profit"].isna()
        & df_check["Net_Sales"].notna()
        & df_check["Estimated_Cost"].notna()
    ),

    "Profit_Margin_Pct": (
        df_check["Profit_Margin_Pct"].isna()
        & df_check["Profit"].notna()
        & df_check["Net_Sales"].notna()
        & df_check["Net_Sales"].ne(0)
    )
}

reconstruction_summary = pd.DataFrame([
    {
        "Column": column,
        "Missing_Values": int(df_check[column].isna().sum()),
        "Recoverable_Values": int(condition.sum()),
        "Not_Recoverable": int(
            df_check[column].isna().sum() - condition.sum()
        )
    }
    for column, condition in reconstruction_checks.items()
])

display(reconstruction_summary)

,Column,Missing_Values,Recoverable_Values,Not_Recoverable
0,Gross_Amount,90,0,90
1,Discount_Amount,238,0,238
2,Net_Sales,238,0,238
3,Profit,238,0,238
4,Profit_Margin_Pct,239,0,239


In [46]:
# Cell 38: Investigate product-level price variation

product_price_analysis = (
    financial_check
    .groupby(["Product_Name", "Category"], dropna=False)
    .agg(
        Total_Records=("Order_ID", "count"),
        Available_Prices=("Unit_Price", "count"),
        Unique_Prices=("Unit_Price", "nunique"),
        Minimum_Price=("Unit_Price", "min"),
        Maximum_Price=("Unit_Price", "max"),
        Median_Price=("Unit_Price", "median")
    )
    .reset_index()
)

product_price_analysis["Price_Varies"] = (
    product_price_analysis["Unique_Prices"] > 1
)

display(
    product_price_analysis.sort_values(
        "Unique_Prices",
        ascending=False
    )
)

,Product_Name,Category,Total_Records,Available_Prices,Unique_Prices,Minimum_Price,Maximum_Price,Median_Price,Price_Varies
96,Shoes,Home Appliances,52,51,51,134.00,7674.70,1483.99,True
53,Novel,Electronics,48,47,47,29.48,5677.47,1026.00,True
110,Skincare,Sports,46,45,45,66.05,19021.37,961.09,True
21,Headphones,Home Appliances,45,44,44,101.04,6293.56,917.80,True
65,Rice,Beauty,44,44,44,56.88,5083.00,874.97,True
...,...,...,...,...,...,...,...,...,...
98,Skincare,Beauty,1,1,1,2113.46,2113.46,2113.46,False
111,Treadmill,Beauty,1,1,1,5709.92,5709.92,5709.92,False
112,Treadmill,Books,1,1,1,870.72,870.72,870.72,False
0,Chair,Books,1,0,0,NaN,NaN,NaN,False


In [47]:
# Cell 39: Investigate Estimated_Cost patterns

cost_analysis = financial_check[
    ["Gross_Amount", "Discount_Amount",
     "Net_Sales", "Estimated_Cost", "Profit"]
].copy()

cost_analysis["Cost_to_NetSales_Ratio"] = (
    cost_analysis["Estimated_Cost"]
    / cost_analysis["Net_Sales"].replace(0, pd.NA)
)

cost_analysis["Cost_to_Gross_Ratio"] = (
    cost_analysis["Estimated_Cost"]
    / cost_analysis["Gross_Amount"].replace(0, pd.NA)
)

print("Estimated Cost Summary:")
display(
    cost_analysis[
        ["Estimated_Cost", "Cost_to_NetSales_Ratio",
         "Cost_to_Gross_Ratio"]
    ].describe().round(4)
)

print("\nCost-to-Net-Sales Ratio Distribution:")
display(
    cost_analysis["Cost_to_NetSales_Ratio"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90])
    .to_frame()
)

Estimated Cost Summary:


,Estimated_Cost
count,2757.0000
mean,3165.4803
std,5385.8133
min,-5446.4800
25%,721.2600
50%,1613.2400
75%,3510.1500
max,109806.2200



Cost-to-Net-Sales Ratio Distribution:


,Cost_to_NetSales_Ratio
count,2756.000000
unique,2756.000000
top,0.877416
freq,1.000000


In [48]:
# Cell 40: Investigate Estimated_Cost anomalies

cost_values = financial_check["Estimated_Cost"].dropna()

print("Negative Cost Records:")
negative_costs = financial_check[
    financial_check["Estimated_Cost"] < 0
]

display(
    negative_costs[
        [
            "Order_ID", "Product_Name", "Category",
            "Quantity", "Gross_Amount", "Net_Sales",
            "Estimated_Cost", "Profit"
        ]
    ]
)

print("\nCost Percentiles:")
display(
    cost_values.quantile(
        [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1]
    ).to_frame("Estimated_Cost")
)

print("\nRecords Above 99th Percentile:")

upper_limit = cost_values.quantile(0.99)

extreme_costs = financial_check[
    financial_check["Estimated_Cost"] > upper_limit
]

print("99th Percentile:", upper_limit)
print("Extreme Records:", len(extreme_costs))

display(
    extreme_costs[
        [
            "Order_ID", "Product_Name", "Category",
            "Quantity", "Net_Sales",
            "Estimated_Cost", "Profit"
        ]
    ].head(15)
)

Negative Cost Records:


,Order_ID,Product_Name,Category,Quantity,Gross_Amount,Net_Sales,Estimated_Cost,Profit
665,ORD100611,Novel,Electronics,5,7535.80,-1507.16,-854.28,-652.88
953,ORD100633,Shoes,Fashion,2,1711.34,-342.27,-254.04,-88.23
1153,ORD101132,Headphones,Beauty,-2,-1742.96,-1662.26,-1055.66,-606.60
1667,ORD102321,Laptop,Sports,-2,-2534.30,-1987.40,-1738.61,-248.79
2015,ORD102085,Skincare,Grocery,-2,-7620.86,-6840.48,-5446.48,-1394.00
2767,ORD101892,Shirt,Grocery,5,2177.25,-435.45,-260.43,-175.02



Cost Percentiles:


,Estimated_Cost
0.00,-5446.4800
0.01,102.8880
0.05,232.7420
0.25,721.2600
0.50,1613.2400
0.75,3510.1500
0.95,11105.4700
0.99,23071.3828
1.00,109806.2200



Records Above 99th Percentile:
99th Percentile: 23071.382800000065
Extreme Records: 28


,Order_ID,Product_Name,Category,Quantity,Net_Sales,Estimated_Cost,Profit
131,ORD102473,Rice,Grocery,5,40227.64,27335.69,12891.95
148,ORD102083,Rice,Electronics,4,34747.03,30027.62,4719.41
330,ORD100311,Shirt,Sports,7,129774.93,109387.04,20387.89
353,ORD101890,Shirt,Furniture,5,47291.53,27882.39,19409.14
758,ORD102036,Treadmill,Books,5,67644.99,55552.36,12092.63
772,ORD102882,Shirt,Fashion,25,45817.48,29659.80,16157.68
813,ORD102063,Mixer,Grocery,5,53493.28,43200.29,10292.99
832,ORD100216,Chair,Sports,4,41171.49,32209.00,8962.49
1050,ORD100484,Novel,Beauty,40,32543.41,24337.56,8205.85
1110,ORD102980,Rice,Sports,6,65405.62,38983.13,26422.49


In [49]:
# Cell 41: Analyze negative quantities and financial values

negative_analysis = {
    "Negative_Quantity": (financial_check["Quantity"] < 0).sum(),
    "Negative_Gross_Amount": (financial_check["Gross_Amount"] < 0).sum(),
    "Negative_Net_Sales": (financial_check["Net_Sales"] < 0).sum(),
    "Negative_Estimated_Cost": (financial_check["Estimated_Cost"] < 0).sum(),
    "Negative_Profit": (financial_check["Profit"] < 0).sum()
}

display(
    pd.DataFrame(
        negative_analysis.items(),
        columns=["Condition", "Record_Count"]
    )
)

print("\nNegative quantity records:")
display(
    financial_check.loc[
        financial_check["Quantity"] < 0,
        [
            "Order_ID", "Quantity", "Return_Status",
            "Order_Status", "Gross_Amount",
            "Net_Sales", "Estimated_Cost", "Profit"
        ]
    ]
)

print("\nNegative sales with positive quantity:")
display(
    financial_check.loc[
        (financial_check["Quantity"] > 0)
        & (financial_check["Net_Sales"] < 0),
        [
            "Order_ID", "Quantity", "Return_Status",
            "Order_Status", "Gross_Amount",
            "Discount_Pct", "Discount_Amount",
            "Net_Sales", "Estimated_Cost", "Profit"
        ]
    ]
)

,Condition,Record_Count
0,Negative_Quantity,3
1,Negative_Gross_Amount,3
2,Negative_Net_Sales,6
3,Negative_Estimated_Cost,6
4,Negative_Profit,6



Negative quantity records:


,Order_ID,Quantity,Return_Status,Order_Status,Gross_Amount,Net_Sales,Estimated_Cost,Profit
1153,ORD101132,-2,No,Pending,-1742.96,-1662.26,-1055.66,-606.60
1667,ORD102321,-2,No,Delivered,-2534.30,-1987.40,-1738.61,-248.79
2015,ORD102085,-2,Yes,Delivered,-7620.86,-6840.48,-5446.48,-1394.00



Negative sales with positive quantity:


,Order_ID,Quantity,Return_Status,Order_Status,Gross_Amount,Discount_Pct,Discount_Amount,Net_Sales,Estimated_Cost,Profit
665,ORD100611,5,No,Delivered,7535.80,120.0,9042.96,-1507.16,-854.28,-652.88
953,ORD100633,2,No,Delivered,1711.34,120.0,2053.61,-342.27,-254.04,-88.23
2767,ORD101892,5,No,Delivered,2177.25,120.0,2612.70,-435.45,-260.43,-175.02


In [50]:
# Cell 42: Identify quantity and discount violations

business_rule_issues = {
    "Negative_Quantity": financial_check["Quantity"] < 0,
    "Zero_Quantity": financial_check["Quantity"] == 0,
    "Negative_Discount": financial_check["Discount_Pct"] < 0,
    "Discount_Above_100": financial_check["Discount_Pct"] > 100
}

rule_summary = pd.DataFrame([
    {
        "Business_Rule": rule,
        "Violation_Count": int(condition.sum())
    }
    for rule, condition in business_rule_issues.items()
])

print("Business Rule Summary:")
display(rule_summary)

# Collect all affected records
issue_mask = pd.DataFrame(
    business_rule_issues,
    index=financial_check.index
).any(axis=1)

issue_records = financial_check.loc[
    issue_mask,
    [
        "Order_ID",
        "Product_Name",
        "Quantity",
        "Discount_Pct",
        "Gross_Amount",
        "Discount_Amount",
        "Net_Sales",
        "Return_Status",
        "Order_Status"
    ]
]

print("Total Records with Quantity/Discount Violations:",
      len(issue_records))

display(issue_records)

Business Rule Summary:


,Business_Rule,Violation_Count
0,Negative_Quantity,3
1,Zero_Quantity,1
2,Negative_Discount,2
3,Discount_Above_100,3


Total Records with Quantity/Discount Violations: 9


,Order_ID,Product_Name,Quantity,Discount_Pct,Gross_Amount,Discount_Amount,Net_Sales,Return_Status,Order_Status
114,ORD101217,Headphones,0,9.04,0.00,0.00,0.00,No,Cancelled
496,ORD102676,Shoes,4,-10.00,1970.80,-197.08,2167.88,No,Delivered
665,ORD100611,Novel,5,120.00,7535.80,9042.96,-1507.16,No,Delivered
953,ORD100633,Shoes,2,120.00,1711.34,2053.61,-342.27,No,Delivered
1153,ORD101132,Headphones,-2,4.63,-1742.96,-80.70,-1662.26,No,Pending
1667,ORD102321,Laptop,-2,21.58,-2534.30,-546.90,-1987.40,No,Delivered
2015,ORD102085,Skincare,-2,10.24,-7620.86,-780.38,-6840.48,Yes,Delivered
2581,ORD102047,Skincare,2,-10.00,262.40,-26.24,288.64,No,Delivered
2767,ORD101892,Shirt,5,120.00,2177.25,2612.70,-435.45,No,Delivered


In [51]:
# Cell 43: Classify business-rule exceptions

df_rule_audit = financial_check.copy()

df_rule_audit["Exception_Reason"] = ""

rules = {
    "Negative Quantity": df_rule_audit["Quantity"] < 0,
    "Zero Quantity": df_rule_audit["Quantity"] == 0,
    "Negative Discount": df_rule_audit["Discount_Pct"] < 0,
    "Discount Above 100%": df_rule_audit["Discount_Pct"] > 100
}

for reason, condition in rules.items():
    df_rule_audit.loc[
        condition,
        "Exception_Reason"
    ] = (
        df_rule_audit.loc[
            condition, "Exception_Reason"
        ].apply(
            lambda existing: (
                existing + "; " if existing else ""
            ) + reason
        )
    )

df_rule_exceptions = df_rule_audit[
    df_rule_audit["Exception_Reason"] != ""
].copy()

print("Total Exception Records:", len(df_rule_exceptions))

display(
    df_rule_exceptions[
        [
            "Order_ID",
            "Quantity",
            "Discount_Pct",
            "Net_Sales",
            "Return_Status",
            "Order_Status",
            "Exception_Reason"
        ]
    ]
)

Total Exception Records: 9


,Order_ID,Quantity,Discount_Pct,Net_Sales,Return_Status,Order_Status,Exception_Reason
114,ORD101217,0,9.04,0.00,No,Cancelled,Zero Quantity
496,ORD102676,4,-10.00,2167.88,No,Delivered,Negative Discount
665,ORD100611,5,120.00,-1507.16,No,Delivered,Discount Above 100%
953,ORD100633,2,120.00,-342.27,No,Delivered,Discount Above 100%
1153,ORD101132,-2,4.63,-1662.26,No,Pending,Negative Quantity
1667,ORD102321,-2,21.58,-1987.40,No,Delivered,Negative Quantity
2015,ORD102085,-2,10.24,-6840.48,Yes,Delivered,Negative Quantity
2581,ORD102047,2,-10.00,288.64,No,Delivered,Negative Discount
2767,ORD101892,5,120.00,-435.45,No,Delivered,Discount Above 100%


In [52]:
# Cell 44: Analyze non-financial missing values

non_financial_columns = [
    "Customer_Age",
    "City",
    "Payment_Method",
    "Shipping_Days",
    "Customer_Rating",
    "Customer_Loyalty_Points",
    "Website_Session_Minutes",
    "Customer_Satisfaction"
]

non_financial_data = df_silver_business[
    non_financial_columns
].replace(r"^\s*$", pd.NA, regex=True)

non_financial_profile = pd.DataFrame({
    "Column": non_financial_columns,
    "Missing_Count": non_financial_data.isna().sum().values,
    "Missing_Percentage": (
        non_financial_data.isna().mean().values * 100
    ).round(2),
    "Unique_Values": [
        non_financial_data[col].nunique()
        for col in non_financial_columns
    ]
})

display(
    non_financial_profile.sort_values(
        "Missing_Count",
        ascending=False
    )
)

,Column,Missing_Count,Missing_Percentage,Unique_Values
4,Customer_Rating,209,6.98,42
7,Customer_Satisfaction,149,4.97,40
0,Customer_Age,120,4.01,74
5,Customer_Loyalty_Points,119,3.97,1479
3,Shipping_Days,105,3.51,14
2,Payment_Method,75,2.50,11
6,Website_Session_Minutes,75,2.50,403
1,City,74,2.47,21


In [53]:
# Cell 45: Analyze non-financial column distributions

numeric_columns = [
    "Customer_Age",
    "Shipping_Days",
    "Customer_Rating",
    "Customer_Loyalty_Points",
    "Website_Session_Minutes",
    "Customer_Satisfaction"
]

categorical_columns = [
    "City",
    "Payment_Method"
]

df_nonfinancial = df_silver_business[
    non_financial_columns
].copy()

# Convert numerical columns
for col in numeric_columns:
    df_nonfinancial[col] = pd.to_numeric(
        df_nonfinancial[col],
        errors="coerce"
    )

print("NUMERICAL COLUMN STATISTICS")

display(
    df_nonfinancial[numeric_columns]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .T
    .round(2)
)

print("\nCATEGORICAL VALUE DISTRIBUTIONS")

for col in categorical_columns:
    print(f"\n{col}")
    display(
        df_nonfinancial[col]
        .value_counts(dropna=False)
        .to_frame("Frequency")
    )

NUMERICAL COLUMN STATISTICS


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Customer_Age,2875.0,35.33,11.36,-13.0,9.00,17.0,28.0,35.0,43.0,53.0,62.00,102.0
Shipping_Days,2890.0,4.56,2.29,-3.0,1.00,1.0,3.0,4.0,6.0,8.0,9.00,35.0
Customer_Rating,2786.0,3.75,0.78,0.0,1.80,2.4,3.2,3.8,4.3,5.0,5.00,9.0
Customer_Loyalty_Points,2876.0,869.79,629.05,0.0,0.00,0.0,355.0,828.0,1295.5,1994.0,2450.00,2970.0
Website_Session_Minutes,2920.0,14.71,9.52,1.0,1.80,3.3,7.9,12.7,19.5,32.7,44.46,96.1
Customer_Satisfaction,2846.0,3.67,0.84,1.0,1.64,2.2,3.1,3.7,4.3,5.0,5.00,5.0



CATEGORICAL VALUE DISTRIBUTIONS

City


,Frequency
City,
Chennai,305
Kochi,295
Coimbatore,291
Delhi,289
Kolkata,286
Hyderabad,284
Mumbai,284
Pune,276
Bengaluru,268



Payment_Method


,Frequency
Payment_Method,
UPI,531
Net Banking,494
Credit Card,481
Debit Card,466
Wallet,459
Cash,454
,75
CASH,11
NET BANKING,11


In [54]:
# Cell 46: Identify numerical data-quality violations

validation_rules = {
    "Customer_Age": {
        "negative": df_nonfinancial["Customer_Age"] < 0,
        "above_100": df_nonfinancial["Customer_Age"] > 100
    },
    "Shipping_Days": {
        "negative": df_nonfinancial["Shipping_Days"] < 0,
        "above_30": df_nonfinancial["Shipping_Days"] > 30
    },
    "Customer_Rating": {
        "below_1": df_nonfinancial["Customer_Rating"] < 1,
        "above_5": df_nonfinancial["Customer_Rating"] > 5
    },
    "Customer_Satisfaction": {
        "below_1": df_nonfinancial["Customer_Satisfaction"] < 1,
        "above_5": df_nonfinancial["Customer_Satisfaction"] > 5
    },
    "Customer_Loyalty_Points": {
        "negative": df_nonfinancial["Customer_Loyalty_Points"] < 0
    },
    "Website_Session_Minutes": {
        "zero_or_negative": df_nonfinancial["Website_Session_Minutes"] <= 0
    }
}

validation_results = []

for column, conditions in validation_rules.items():
    for rule, condition in conditions.items():
        validation_results.append({
            "Column": column,
            "Rule": rule,
            "Violation_Count": int(condition.sum())
        })

validation_summary = pd.DataFrame(validation_results)

display(validation_summary)

,Column,Rule,Violation_Count
0,Customer_Age,negative,2
1,Customer_Age,above_100,4
2,Shipping_Days,negative,2
3,Shipping_Days,above_30,4
4,Customer_Rating,below_1,3
5,Customer_Rating,above_5,4
6,Customer_Satisfaction,below_1,0
7,Customer_Satisfaction,above_5,0
8,Customer_Loyalty_Points,negative,0
9,Website_Session_Minutes,zero_or_negative,0


In [55]:
# Cell 47: Identify categorical values requiring standardization

for column in categorical_columns:

    print(f"\nOriginal values in {column}:")

    values = (
        df_nonfinancial[column]
        .astype("string")
        .str.strip()
    )

    print(
        values.value_counts(dropna=False).to_string()
    )


Original values in City:
City
Chennai       305
Kochi         295
Coimbatore    291
Delhi         289
Kolkata       286
Hyderabad     284
Mumbai        284
Pune          276
Bengaluru     268
Jaipur        253
               74
bengaluru      15
chennai        13
pune           10
mumbai         10
hyderabad       9
kochi           8
kolkata         8
delhi           7
coimbatore      7
jaipur          2
nan             1

Original values in Payment_Method:
Payment_Method
UPI            531
Net Banking    494
Credit Card    481
Debit Card     466
Wallet         459
Cash           454
                75
CASH            11
NET BANKING     11
WALLET           5
CREDIT CARD      5
DEBIT CARD       3


In [56]:
# Cell 48: Inspect invalid numerical records

invalid_conditions = (
    (df_nonfinancial["Customer_Age"] < 0)
    | (df_nonfinancial["Customer_Age"] > 100)
    | (df_nonfinancial["Shipping_Days"] < 0)
    | (df_nonfinancial["Shipping_Days"] > 30)
    | (df_nonfinancial["Customer_Rating"] < 1)
    | (df_nonfinancial["Customer_Rating"] > 5)
)

invalid_records = df_silver_business.loc[
    invalid_conditions,
    [
        "Order_ID",
        "Customer_ID",
        "Customer_Age",
        "Shipping_Days",
        "Customer_Rating",
        "Customer_Satisfaction",
        "Order_Status",
        "Return_Status"
    ]
].copy()

print("Total affected records:", len(invalid_records))

display(invalid_records)

Total affected records: 19


,Order_ID,Customer_ID,Customer_Age,Shipping_Days,Customer_Rating,Customer_Satisfaction,Order_Status,Return_Status
74,ORD102695,C1470,-13.0,3.0,4.7,4.2,Delivered,No
169,ORD101066,C1276,,35.0,3.5,,Cancelled,No
423,ORD102987,C1732,32.0,35.0,3.7,2.1,Delivered,No
973,ORD102322,C1673,44.0,7.0,0.0,4.2,Delivered,No
1002,ORD101500,C1277,41.0,1.0,6.0,3.6,Pending,No
1103,ORD102616,C1585,39.0,1.0,0.0,2.8,Delayed,No
1407,ORD100572,C1674,37.0,5.0,9.0,3.9,Pending,No
1424,ORD102989,C1694,54.0,-3.0,,3.5,Pending,No
1425,ORD101138,C1208,33.0,5.0,6.0,,Delivered,No
1438,ORD101895,C1386,32.0,35.0,3.2,2.3,Delivered,No


In [58]:
# Cell 49: Correct invalid numerical values in working copy

df_silver_clean = df_silver_business.copy()

numeric_validation_columns = [
    "Customer_Age",
    "Shipping_Days",
    "Customer_Rating",
    "Customer_Loyalty_Points",
    "Website_Session_Minutes",
    "Customer_Satisfaction"
]

for col in numeric_validation_columns:
    df_silver_clean[col] = pd.to_numeric(
        df_silver_clean[col],
        errors="coerce"
    )

# Customer age: provisional valid range 0–100
df_silver_clean.loc[
    (df_silver_clean["Customer_Age"] < 0)
    | (df_silver_clean["Customer_Age"] > 100),
    "Customer_Age"
] = pd.NA

# Shipping days: negative values are invalid
df_silver_clean.loc[
    df_silver_clean["Shipping_Days"] < 0,
    "Shipping_Days"
] = pd.NA

# Customer rating: valid scale 1–5
df_silver_clean.loc[
    (df_silver_clean["Customer_Rating"] < 1)
    | (df_silver_clean["Customer_Rating"] > 5),
    "Customer_Rating"
] = pd.NA

print("Remaining invalid values:")

display(pd.DataFrame({
    "Customer_Age_Invalid": [
        ((df_silver_clean["Customer_Age"] < 0) |
         (df_silver_clean["Customer_Age"] > 100)).sum()
    ],
    "Shipping_Days_Negative": [
        (df_silver_clean["Shipping_Days"] < 0).sum()
    ],
    "Customer_Rating_Invalid": [
        ((df_silver_clean["Customer_Rating"] < 1) |
         (df_silver_clean["Customer_Rating"] > 5)).sum()
    ]
}))

print("Missing values after validation:")
display(
    df_silver_clean[numeric_validation_columns]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

Remaining invalid values:


,Customer_Age_Invalid,Shipping_Days_Negative,Customer_Rating_Invalid
0,0,0,0


Missing values after validation:


,Missing_Count
Customer_Age,126
Shipping_Days,107
Customer_Rating,216
Customer_Loyalty_Points,119
Website_Session_Minutes,75
Customer_Satisfaction,149


In [59]:
# Cell 50: Standardize categorical values

categorical_clean_columns = [
    "City",
    "Payment_Method"
]

missing_tokens = {
    "", "nan", "null", "none", "n/a", "na", "nat"
}

for col in categorical_clean_columns:

    df_silver_clean[col] = (
        df_silver_clean[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    df_silver_clean[col] = (
        df_silver_clean[col]
        .mask(df_silver_clean[col].isin(missing_tokens))
        .str.title()
    )

print("City values after standardization:")
display(df_silver_clean["City"].value_counts(dropna=False))

print("\nPayment Method values after standardization:")
display(
    df_silver_clean["Payment_Method"]
    .value_counts(dropna=False)
)

City values after standardization:


City
Chennai       318
Kochi         303
Coimbatore    298
Delhi         296
Kolkata       294
Mumbai        294
Hyderabad     293
Pune          286
Bengaluru     283
Jaipur        255
<NA>           75
Name: count, dtype: int64[pyarrow]


Payment Method values after standardization:


Payment_Method
Upi            531
Net Banking    505
Credit Card    486
Debit Card     469
Cash           465
Wallet         464
<NA>            75
Name: count, dtype: int64[pyarrow]

In [60]:
# Cell 51: Analyze customer-level attribute consistency

customer_attributes = [
    "Customer_Age",
    "Customer_Loyalty_Points",
    "City"
]

customer_consistency = (
    df_silver_clean
    .groupby("Customer_ID")[customer_attributes]
    .nunique(dropna=True)
    .reset_index()
)

for col in customer_attributes:
    print(
        f"{col} - Customers with multiple values:",
        (customer_consistency[col] > 1).sum()
    )

print("\nCustomer attribute consistency summary:")
display(
    customer_consistency[
        customer_attributes
    ].describe()
)

Customer_Age - Customers with multiple values: 690
Customer_Loyalty_Points - Customers with multiple values: 692
City - Customers with multiple values: 688

Customer attribute consistency summary:


,Customer_Age,Customer_Loyalty_Points,City
count,776.000000,776.000000,776.000000
mean,3.524485,3.619845,3.149485
std,1.667734,1.726774,1.333760
min,0.000000,0.000000,0.000000
25%,2.000000,2.000000,2.000000
50%,3.000000,3.000000,3.000000
75%,5.000000,5.000000,4.000000
max,10.000000,10.000000,7.000000


In [61]:
# Cell 52: Inspect customer-level attribute variations

customer_variation = (
    df_silver_clean
    .groupby("Customer_ID")
    .agg(
        Order_Count=("Order_ID", "count"),
        Age_Variations=("Customer_Age", "nunique"),
        Loyalty_Variations=("Customer_Loyalty_Points", "nunique"),
        City_Variations=("City", "nunique")
    )
    .reset_index()
)

# Customers with multiple values across attributes
variable_customers = customer_variation[
    (customer_variation["Age_Variations"] > 1)
    | (customer_variation["Loyalty_Variations"] > 1)
    | (customer_variation["City_Variations"] > 1)
]

print("Customers with attribute variations:", len(variable_customers))

display(
    variable_customers.sort_values(
        "Order_Count",
        ascending=False
    ).head(10)
)

# Inspect transactions for the first customer
sample_customer = variable_customers.iloc[0]["Customer_ID"]

print("\nSample Customer:", sample_customer)

display(
    df_silver_clean.loc[
        df_silver_clean["Customer_ID"] == sample_customer,
        [
            "Order_ID",
            "Order_Date",
            "Customer_Age",
            "Customer_Loyalty_Points",
            "City"
        ]
    ].sort_values("Order_Date")
)

Customers with attribute variations: 704


,Customer_ID,Order_Count,Age_Variations,Loyalty_Variations,City_Variations
721,C1744,10,10,9,6
402,C1418,10,9,10,6
360,C1375,9,7,8,7
407,C1423,9,5,5,5
236,C1247,9,9,8,6
765,C1790,9,8,9,5
130,C1136,9,7,7,7
270,C1282,9,7,9,6
768,C1793,9,6,7,5
51,C1054,8,7,8,6



Sample Customer: C1001


,Order_ID,Order_Date,Customer_Age,Customer_Loyalty_Points,City
316,ORD100674,2025-02-04,17.0,1629.0,Hyderabad
148,ORD102083,2025-04-29,30.0,1240.0,Bengaluru
871,ORD102760,2025-06-06,32.0,97.0,Coimbatore


In [62]:
# Cell 53: Analyze missingness across business groups

group_columns = [
    "Customer_Segment",
    "Order_Status",
    "Region",
    "Category"
]

target_columns = [
    "Customer_Age",
    "Shipping_Days",
    "Customer_Rating",
    "Customer_Loyalty_Points",
    "Website_Session_Minutes",
    "Customer_Satisfaction"
]

for group_col in group_columns:

    print(f"\nMissing-value analysis by {group_col}")

    group_analysis = (
        df_silver_clean
        .groupby(group_col, dropna=False)[target_columns]
        .agg(lambda x: x.isna().sum())
    )

    display(group_analysis)


Missing-value analysis by Customer_Segment


,Customer_Age,Shipping_Days,Customer_Rating,Customer_Loyalty_Points,Website_Session_Minutes,Customer_Satisfaction
Customer_Segment,,,,,,
New,19,13,31,16,7,18
Occasional,28,25,46,31,15,36
Premium,27,11,38,18,15,28
Regular,52,58,101,54,38,67



Missing-value analysis by Order_Status


,Customer_Age,Shipping_Days,Customer_Rating,Customer_Loyalty_Points,Website_Session_Minutes,Customer_Satisfaction
Order_Status,,,,,,
Cancelled,7,5,14,8,5,17
Delayed,9,6,14,12,7,14
Delivered,89,83,160,89,55,91
Pending,13,5,12,5,5,10
Returned,8,8,16,5,3,17



Missing-value analysis by Region


,Customer_Age,Shipping_Days,Customer_Rating,Customer_Loyalty_Points,Website_Session_Minutes,Customer_Satisfaction
Region,,,,,,
Central,23,29,60,21,15,29
East,21,24,38,27,17,35
North,31,18,36,21,16,28
South,22,20,41,26,15,18
West,29,16,41,24,12,39



Missing-value analysis by Category


,Customer_Age,Shipping_Days,Customer_Rating,Customer_Loyalty_Points,Website_Session_Minutes,Customer_Satisfaction
Category,,,,,,
Beauty,0,0,1,0,0,0
Books,0,1,0,0,0,1
Electronics,0,1,0,1,0,1
Fashion,0,0,0,0,1,0
Furniture,0,0,1,0,0,0
Grocery,0,0,1,0,0,0
Home Appliances,0,0,0,0,0,1
Sports,0,0,2,1,0,2
Beauty,12,14,27,16,7,16


In [63]:
# Cell 54: Calculate missing percentages by business groups

group_columns = [
    "Customer_Segment",
    "Order_Status",
    "Region",
    "Category"
]

target_columns = [
    "Customer_Age",
    "Shipping_Days",
    "Customer_Rating",
    "Customer_Loyalty_Points",
    "Website_Session_Minutes",
    "Customer_Satisfaction"
]

for group_col in group_columns:

    print(f"\nMissing-value rates by {group_col}")

    grouped = df_silver_clean.groupby(
        group_col,
        dropna=False
    )

    group_sizes = grouped.size().rename("Group_Size")

    missing_counts = grouped[target_columns].agg(
        lambda x: x.isna().sum()
    )

    missing_rates = (
        missing_counts
        .div(group_sizes, axis=0)
        .mul(100)
        .round(2)
    )

    result = pd.concat(
        [group_sizes, missing_counts, missing_rates.add_suffix("_Missing_Pct")],
        axis=1
    )

    display(result)

print("\nCategory values with exact representations:")

for value in df_silver_clean["Category"].dropna().unique():
    print(repr(value))


Missing-value rates by Customer_Segment


,Group_Size,Customer_Age,Shipping_Days,Customer_Rating,Customer_Loyalty_Points,Website_Session_Minutes,Customer_Satisfaction,Customer_Age_Missing_Pct,Shipping_Days_Missing_Pct,Customer_Rating_Missing_Pct,Customer_Loyalty_Points_Missing_Pct,Website_Session_Minutes_Missing_Pct,Customer_Satisfaction_Missing_Pct
Customer_Segment,,,,,,,,,,,,,
New,372,19,13,31,16,7,18,5.11,3.49,8.33,4.30,1.88,4.84
Occasional,644,28,25,46,31,15,36,4.35,3.88,7.14,4.81,2.33,5.59
Premium,523,27,11,38,18,15,28,5.16,2.10,7.27,3.44,2.87,5.35
Regular,1456,52,58,101,54,38,67,3.57,3.98,6.94,3.71,2.61,4.60



Missing-value rates by Order_Status


,Group_Size,Customer_Age,Shipping_Days,Customer_Rating,Customer_Loyalty_Points,Website_Session_Minutes,Customer_Satisfaction,Customer_Age_Missing_Pct,Shipping_Days_Missing_Pct,Customer_Rating_Missing_Pct,Customer_Loyalty_Points_Missing_Pct,Website_Session_Minutes_Missing_Pct,Customer_Satisfaction_Missing_Pct
Order_Status,,,,,,,,,,,,,
Cancelled,172,7,5,14,8,5,17,4.07,2.91,8.14,4.65,2.91,9.88
Delayed,175,9,6,14,12,7,14,5.14,3.43,8.00,6.86,4.00,8.00
Delivered,2163,89,83,160,89,55,91,4.11,3.84,7.40,4.11,2.54,4.21
Pending,258,13,5,12,5,5,10,5.04,1.94,4.65,1.94,1.94,3.88
Returned,227,8,8,16,5,3,17,3.52,3.52,7.05,2.20,1.32,7.49



Missing-value rates by Region


,Group_Size,Customer_Age,Shipping_Days,Customer_Rating,Customer_Loyalty_Points,Website_Session_Minutes,Customer_Satisfaction,Customer_Age_Missing_Pct,Shipping_Days_Missing_Pct,Customer_Rating_Missing_Pct,Customer_Loyalty_Points_Missing_Pct,Website_Session_Minutes_Missing_Pct,Customer_Satisfaction_Missing_Pct
Region,,,,,,,,,,,,,
Central,611,23,29,60,21,15,29,3.76,4.75,9.82,3.44,2.45,4.75
East,623,21,24,38,27,17,35,3.37,3.85,6.10,4.33,2.73,5.62
North,586,31,18,36,21,16,28,5.29,3.07,6.14,3.58,2.73,4.78
South,578,22,20,41,26,15,18,3.81,3.46,7.09,4.50,2.60,3.11
West,597,29,16,41,24,12,39,4.86,2.68,6.87,4.02,2.01,6.53



Missing-value rates by Category


,Group_Size,Customer_Age,Shipping_Days,Customer_Rating,Customer_Loyalty_Points,Website_Session_Minutes,Customer_Satisfaction,Customer_Age_Missing_Pct,Shipping_Days_Missing_Pct,Customer_Rating_Missing_Pct,Customer_Loyalty_Points_Missing_Pct,Website_Session_Minutes_Missing_Pct,Customer_Satisfaction_Missing_Pct
Category,,,,,,,,,,,,,
Beauty,4,0,0,1,0,0,0,0.00,0.00,25.00,0.00,0.00,0.00
Books,10,0,1,0,0,0,1,0.00,10.00,0.00,0.00,0.00,10.00
Electronics,7,0,1,0,1,0,1,0.00,14.29,0.00,14.29,0.00,14.29
Fashion,7,0,0,0,0,1,0,0.00,0.00,0.00,0.00,14.29,0.00
Furniture,5,0,0,1,0,0,0,0.00,0.00,20.00,0.00,0.00,0.00
Grocery,8,0,0,1,0,0,0,0.00,0.00,12.50,0.00,0.00,0.00
Home Appliances,10,0,0,0,0,0,1,0.00,0.00,0.00,0.00,0.00,10.00
Sports,9,0,0,2,1,0,2,0.00,0.00,22.22,11.11,0.00,22.22
Beauty,357,12,14,27,16,7,16,3.36,3.92,7.56,4.48,1.96,4.48



Category values with exact representations:
'Sports'
'Home Appliances'
'Electronics'
'Fashion'
'Beauty'
'Grocery'
'Books'
'Furniture'
' Books '
' Beauty '
' Sports '
' Home Appliances '
' Grocery '
' Electronics '
' Fashion '
' Furniture '


In [64]:
# Cell 55: Standardize Category values

df_silver_clean["Category"] = (
    df_silver_clean["Category"]
    .astype("string")
    .str.strip()
    .str.title()
)

print("Standardized Category values:")
display(
    df_silver_clean["Category"]
    .value_counts(dropna=False)
)

print(
    "\nUnique Categories:",
    df_silver_clean["Category"].nunique()
)

Standardized Category values:


Category
Home Appliances    429
Sports             385
Grocery            376
Electronics        371
Furniture          363
Beauty             361
Fashion            357
Books              353
Name: count, dtype: int64[pyarrow]


Unique Categories: 8


In [65]:
# Cell 56: Apply documented numerical imputation strategies

imputation_groups = {
    "Customer_Age": "Customer_Segment",
    "Shipping_Days": "Order_Status",
    "Customer_Rating": "Order_Status",
    "Customer_Satisfaction": "Customer_Segment"
}

global_medians = {
    col: df_silver_clean[col].median()
    for col in [
        "Customer_Age",
        "Shipping_Days",
        "Customer_Rating",
        "Customer_Loyalty_Points",
        "Website_Session_Minutes",
        "Customer_Satisfaction"
    ]
}

imputation_audit = []

for column, group_column in imputation_groups.items():

    missing_before = int(df_silver_clean[column].isna().sum())

    group_medians = (
        df_silver_clean
        .groupby(group_column)[column]
        .transform("median")
    )

    df_silver_clean[column] = (
        df_silver_clean[column]
        .fillna(group_medians)
        .fillna(global_medians[column])
    )

    imputation_audit.append({
        "Column": column,
        "Missing_Before": missing_before,
        "Imputed_Values": missing_before - int(
            df_silver_clean[column].isna().sum()
        ),
        "Missing_After": int(
            df_silver_clean[column].isna().sum()
        ),
        "Method": f"Median by {group_column}, global fallback"
    })

# Global median imputation for remaining numeric columns
for column in [
    "Customer_Loyalty_Points",
    "Website_Session_Minutes"
]:

    missing_before = int(df_silver_clean[column].isna().sum())

    df_silver_clean[column] = (
        df_silver_clean[column]
        .fillna(global_medians[column])
    )

    imputation_audit.append({
        "Column": column,
        "Missing_Before": missing_before,
        "Imputed_Values": missing_before - int(
            df_silver_clean[column].isna().sum()
        ),
        "Missing_After": int(
            df_silver_clean[column].isna().sum()
        ),
        "Method": "Global median"
    })

imputation_audit_df = pd.DataFrame(imputation_audit)

display(imputation_audit_df)

print("\nRemaining numerical missing values:")
display(
    df_silver_clean[numeric_validation_columns]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

,Column,Missing_Before,Imputed_Values,Missing_After,Method
0,Customer_Age,126,126,0,"Median by Customer_Segment, global fallback"
1,Shipping_Days,107,107,0,"Median by Order_Status, global fallback"
2,Customer_Rating,216,216,0,"Median by Order_Status, global fallback"
3,Customer_Satisfaction,149,149,0,"Median by Customer_Segment, global fallback"
4,Customer_Loyalty_Points,119,119,0,Global median
5,Website_Session_Minutes,75,75,0,Global median



Remaining numerical missing values:


,Missing_Count
Customer_Age,0
Shipping_Days,0
Customer_Rating,0
Customer_Loyalty_Points,0
Website_Session_Minutes,0
Customer_Satisfaction,0


In [66]:
# Cell 57: Restore data and reapply validated cleaning rules

df_silver_clean = df_silver_business.copy()

# 1. Convert numerical columns
numeric_columns = [
    "Customer_Age",
    "Shipping_Days",
    "Customer_Rating",
    "Customer_Loyalty_Points",
    "Website_Session_Minutes",
    "Customer_Satisfaction"
]

for col in numeric_columns:
    df_silver_clean[col] = pd.to_numeric(
        df_silver_clean[col], errors="coerce"
    )

# 2. Handle invalid numerical values
df_silver_clean.loc[
    (df_silver_clean["Customer_Age"] < 0) |
    (df_silver_clean["Customer_Age"] > 100),
    "Customer_Age"
] = pd.NA

df_silver_clean.loc[
    df_silver_clean["Shipping_Days"] < 0,
    "Shipping_Days"
] = pd.NA

df_silver_clean.loc[
    ~df_silver_clean["Customer_Rating"].between(1, 5),
    "Customer_Rating"
] = pd.NA

# 3. Standardize categorical columns
for col in ["City", "Payment_Method", "Category"]:
    df_silver_clean[col] = (
        df_silver_clean[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace(["", "nan", "null", "none", "n/a", "na", "nat"], pd.NA)
        .str.title()
    )

# 4. Impute Customer_Age only
age_medians = (
    df_silver_clean.groupby("Customer_Segment")["Customer_Age"]
    .transform("median")
)

global_age_median = df_silver_clean["Customer_Age"].median()

df_silver_clean["Customer_Age"] = (
    df_silver_clean["Customer_Age"]
    .fillna(age_medians)
    .fillna(global_age_median)
)

# 5. Review remaining missing values
print("Missing values after revised cleaning:\n")

display(
    df_silver_clean[numeric_columns]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

print("\nCategory count:", df_silver_clean["Category"].nunique())

Missing values after revised cleaning:



,Missing_Count
Customer_Age,0
Shipping_Days,107
Customer_Rating,216
Customer_Loyalty_Points,119
Website_Session_Minutes,75
Customer_Satisfaction,149



Category count: 8


In [67]:
# Cell 58: Handle missing categorical values

categorical_columns = ["City", "Payment_Method"]

for col in categorical_columns:
    df_silver_clean[col] = (
        df_silver_clean[col]
        .fillna("Unknown")
    )

print("Missing categorical values:")

display(
    df_silver_clean[categorical_columns]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

print("\nCity distribution:")
display(df_silver_clean["City"].value_counts())

print("\nPayment Method distribution:")
display(df_silver_clean["Payment_Method"].value_counts())

Missing categorical values:


,Missing_Count
City,0
Payment_Method,0



City distribution:


City
Chennai       318
Kochi         303
Coimbatore    298
Delhi         296
Kolkata       294
Mumbai        294
Hyderabad     293
Pune          286
Bengaluru     283
Jaipur        255
Unknown        75
Name: count, dtype: int64[pyarrow]


Payment Method distribution:


Payment_Method
Upi            531
Net Banking    505
Credit Card    486
Debit Card     469
Cash           465
Wallet         464
Unknown         75
Name: count, dtype: int64[pyarrow]

In [69]:
# Cell 59: Shipping Days missing-value analysis

shipping_analysis = (
    df_silver_clean
    .groupby("Order_Status", dropna=False)
    .agg(
        Total_Orders=("Order_ID", "count"),
        Missing_Shipping=("Shipping_Days", lambda x: x.isna().sum()),
        Available_Shipping=("Shipping_Days", "count"),
        Average_Shipping=("Shipping_Days", "mean"),
        Median_Shipping=("Shipping_Days", "median")
    )
)

shipping_analysis["Missing_Percentage"] = (
    shipping_analysis["Missing_Shipping"]
    / shipping_analysis["Total_Orders"] * 100
).round(2)

display(shipping_analysis)

,Total_Orders,Missing_Shipping,Available_Shipping,Average_Shipping,Median_Shipping,Missing_Percentage
Order_Status,,,,,,
Cancelled,172,5,167,4.730539,5.0,2.91
Delayed,175,6,169,4.408284,4.0,3.43
Delivered,2163,83,2080,4.553846,4.0,3.84
Pending,258,5,253,4.636364,5.0,1.94
Returned,227,8,219,4.584475,5.0,3.52


In [70]:
# Cell 60: Flag missing shipping information

df_silver_clean["Shipping_Data_Status"] = "Available"

df_silver_clean.loc[
    df_silver_clean["Shipping_Days"].isna(),
    "Shipping_Data_Status"
] = "Missing"

print("Shipping data-quality summary:")

display(
    df_silver_clean["Shipping_Data_Status"]
    .value_counts()
    .to_frame("Record_Count")
)

print("\nShipping status breakdown:")

display(
    pd.crosstab(
        df_silver_clean["Order_Status"],
        df_silver_clean["Shipping_Data_Status"]
    )
)

Shipping data-quality summary:


,Record_Count
Shipping_Data_Status,
Available,2888
Missing,107



Shipping status breakdown:


Shipping_Data_Status,Available,Missing
Order_Status,,
Cancelled,167,5
Delayed,169,6
Delivered,2080,83
Pending,253,5
Returned,219,8


In [71]:
# Cell 61: Final numerical validation

validation_results = {
    "Negative Customer_Age": (
        df_silver_clean["Customer_Age"] < 0
    ).sum(),

    "Customer_Age Above 100": (
        df_silver_clean["Customer_Age"] > 100
    ).sum(),

    "Negative Shipping_Days": (
        df_silver_clean["Shipping_Days"] < 0
    ).sum(),

    "Invalid Customer_Rating": (
        ~df_silver_clean["Customer_Rating"].between(1, 5)
    ).sum(),

    "Invalid Customer_Satisfaction": (
        ~df_silver_clean["Customer_Satisfaction"].between(1, 5)
    ).sum(),

    "Negative Customer_Loyalty_Points": (
        df_silver_clean["Customer_Loyalty_Points"] < 0
    ).sum(),

    "Non-positive Website_Session_Minutes": (
        df_silver_clean["Website_Session_Minutes"] <= 0
    ).sum()
}

display(
    pd.DataFrame(
        validation_results.items(),
        columns=["Validation_Check", "Violation_Count"]
    )
)

print("Total records:", len(df_silver_clean))
print("Duplicate Order IDs:", df_silver_clean["Order_ID"].duplicated().sum())

,Validation_Check,Violation_Count
0,Negative Customer_Age,0
1,Customer_Age Above 100,0
2,Negative Shipping_Days,0
3,Invalid Customer_Rating,216
4,Invalid Customer_Satisfaction,149
5,Negative Customer_Loyalty_Points,0
6,Non-positive Website_Session_Minutes,0


Total records: 2995
Duplicate Order IDs: 0


In [72]:
# Cell 62: Separate invalid values from missing values

rating_invalid = (
    df_silver_clean["Customer_Rating"].notna()
    & ~df_silver_clean["Customer_Rating"].between(1, 5)
)

satisfaction_invalid = (
    df_silver_clean["Customer_Satisfaction"].notna()
    & ~df_silver_clean["Customer_Satisfaction"].between(1, 5)
)

validation_summary = pd.DataFrame({
    "Column": [
        "Customer_Rating",
        "Customer_Satisfaction"
    ],
    "Invalid_Values": [
        rating_invalid.sum(),
        satisfaction_invalid.sum()
    ],
    "Missing_Values": [
        df_silver_clean["Customer_Rating"].isna().sum(),
        df_silver_clean["Customer_Satisfaction"].isna().sum()
    ]
})

display(validation_summary)

,Column,Invalid_Values,Missing_Values
0,Customer_Rating,0,216
1,Customer_Satisfaction,0,149


In [74]:
# Cell 63: Convert financial columns to numeric safely

financial_columns = [
    "Quantity",
    "Unit_Price",
    "Discount_Pct",
    "Gross_Amount",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

for col in financial_columns:
    df_silver_clean[col] = pd.to_numeric(
        df_silver_clean[col],
        errors="coerce"
    )

print("Financial column data types:\n")
display(df_silver_clean[financial_columns].dtypes)

print("\nFinancial missing-value counts:\n")
display(df_silver_clean[financial_columns].isna().sum())

Financial column data types:



Quantity               int64
Unit_Price           float64
Discount_Pct         float64
Gross_Amount         float64
Discount_Amount      float64
Net_Sales            float64
Estimated_Cost       float64
Profit               float64
Profit_Margin_Pct    float64
dtype: object


Financial missing-value counts:



Quantity               0
Unit_Price            90
Discount_Pct         149
Gross_Amount          90
Discount_Amount      238
Net_Sales            238
Estimated_Cost       238
Profit               238
Profit_Margin_Pct    239
dtype: int64

In [75]:
# Cell 64: Financial business-rule validation

financial_checks = {
    "Negative Quantity": df_silver_clean["Quantity"] < 0,
    "Zero Quantity": df_silver_clean["Quantity"] == 0,
    "Negative Discount": df_silver_clean["Discount_Pct"] < 0,
    "Discount Above 100%": df_silver_clean["Discount_Pct"] > 100,
    "Negative Gross Amount": df_silver_clean["Gross_Amount"] < 0,
    "Negative Net Sales": df_silver_clean["Net_Sales"] < 0,
    "Negative Estimated Cost": df_silver_clean["Estimated_Cost"] < 0,
    "Negative Profit": df_silver_clean["Profit"] < 0
}

financial_exception_summary = pd.DataFrame([
    {
        "Exception": name,
        "Count": int(condition.sum())
    }
    for name, condition in financial_checks.items()
])

display(financial_exception_summary)

,Exception,Count
0,Negative Quantity,3
1,Zero Quantity,1
2,Negative Discount,2
3,Discount Above 100%,3
4,Negative Gross Amount,3
5,Negative Net Sales,6
6,Negative Estimated Cost,6
7,Negative Profit,6


In [76]:
# Cell 65: Create financial exception audit

exception_conditions = {
    "Negative Quantity": df_silver_clean["Quantity"] < 0,
    "Zero Quantity": df_silver_clean["Quantity"] == 0,
    "Negative Discount": df_silver_clean["Discount_Pct"] < 0,
    "Discount Above 100%": df_silver_clean["Discount_Pct"] > 100,
    "Negative Gross Amount": df_silver_clean["Gross_Amount"] < 0,
    "Negative Net Sales": df_silver_clean["Net_Sales"] < 0,
    "Negative Estimated Cost": df_silver_clean["Estimated_Cost"] < 0,
    "Negative Profit": df_silver_clean["Profit"] < 0
}

exception_mask = pd.Series(False, index=df_silver_clean.index)

for condition in exception_conditions.values():
    exception_mask |= condition.fillna(False)

df_financial_exceptions = df_silver_clean.loc[
    exception_mask,
    [
        "Order_ID", "Quantity", "Discount_Pct",
        "Gross_Amount", "Discount_Amount",
        "Net_Sales", "Estimated_Cost", "Profit",
        "Return_Status", "Order_Status"
    ]
].copy()

df_financial_exceptions["Exception_Reason"] = [
    "; ".join(
        reason for reason, condition in exception_conditions.items()
        if bool(condition.loc[idx]) 
    )
    for idx in df_financial_exceptions.index
]

print("Unique exception records:", len(df_financial_exceptions))

display(df_financial_exceptions)


Unique exception records: 9


,Order_ID,Quantity,Discount_Pct,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Return_Status,Order_Status,Exception_Reason
114,ORD101217,0,9.04,0.00,0.00,0.00,0.00,0.00,No,Cancelled,Zero Quantity
496,ORD102676,4,-10.00,1970.80,-197.08,2167.88,1681.25,486.63,No,Delivered,Negative Discount
665,ORD100611,5,120.00,7535.80,9042.96,-1507.16,-854.28,-652.88,No,Delivered,Discount Above 100%; Negative Net Sales; Negat...
953,ORD100633,2,120.00,1711.34,2053.61,-342.27,-254.04,-88.23,No,Delivered,Discount Above 100%; Negative Net Sales; Negat...
1153,ORD101132,-2,4.63,-1742.96,-80.70,-1662.26,-1055.66,-606.60,No,Pending,Negative Quantity; Negative Gross Amount; Nega...
1667,ORD102321,-2,21.58,-2534.30,-546.90,-1987.40,-1738.61,-248.79,No,Delivered,Negative Quantity; Negative Gross Amount; Nega...
2015,ORD102085,-2,10.24,-7620.86,-780.38,-6840.48,-5446.48,-1394.00,Yes,Delivered,Negative Quantity; Negative Gross Amount; Nega...
2581,ORD102047,2,-10.00,262.40,-26.24,288.64,242.52,46.12,No,Delivered,Negative Discount
2767,ORD101892,5,120.00,2177.25,2612.70,-435.45,-260.43,-175.02,No,Delivered,Discount Above 100%; Negative Net Sales; Negat...


In [78]:
# Cell 66: Save financial exception audit safely

from pathlib import Path

# Locate project root (one level above the python folder)
project_root = Path.cwd().parent

# Create report directory if it doesn't exist
report_dir = project_root / "reports" / "data_quality"
report_dir.mkdir(parents=True, exist_ok=True)

# Save exception report
exception_report_path = report_dir / "financial_exception_audit.csv"

df_financial_exceptions.to_csv(
    exception_report_path,
    index=False
)

print("Financial exception audit saved successfully!")
print("Records:", len(df_financial_exceptions))
print("File:", exception_report_path.resolve())

Financial exception audit saved successfully!
Records: 9
File: C:\Calibo_Course\Power BI\reports\data_quality\financial_exception_audit.csv


In [79]:
from pathlib import Path

print("Current notebook directory:")
print(Path.cwd())

print("\nExpected project directory:")
project_root = Path(
    r"C:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project"
)
print(project_root)

print("\nProject directory exists:", project_root.exists())
print("Reports directory exists:", (project_root / "reports").exists())

Current notebook directory:
c:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project

Expected project directory:
C:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project

Project directory exists: True
Reports directory exists: True


In [80]:
from pathlib import Path

project_root = Path.cwd()

report_dir = project_root / "reports" / "data_quality"
report_dir.mkdir(parents=True, exist_ok=True)

exception_report_path = (
    report_dir / "financial_exception_audit.csv"
)

df_financial_exceptions.to_csv(
    exception_report_path,
    index=False
)

print("Saved:", exception_report_path.resolve())
print("File exists:", exception_report_path.exists())
print("Records:", len(pd.read_csv(exception_report_path)))

Saved: C:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\reports\data_quality\financial_exception_audit.csv
File exists: True
Records: 9


In [81]:
# Cell 69: Validate financial calculation consistency

formula_checks = {
    "Gross_Amount": (
        df_silver_clean["Quantity"] * df_silver_clean["Unit_Price"]
    ),

    "Discount_Amount": (
        df_silver_clean["Gross_Amount"]
        * df_silver_clean["Discount_Pct"] / 100
    ),

    "Net_Sales": (
        df_silver_clean["Gross_Amount"]
        - df_silver_clean["Discount_Amount"]
    ),

    "Profit": (
        df_silver_clean["Net_Sales"]
        - df_silver_clean["Estimated_Cost"]
    ),

    "Profit_Margin_Pct": (
        df_silver_clean["Profit"]
        / df_silver_clean["Net_Sales"] * 100
    )
}

formula_results = []

for column, calculated_values in formula_checks.items():

    valid_rows = (
        df_silver_clean[column].notna()
        & calculated_values.notna()
        & calculated_values.abs().ne(float("inf"))
    )

    differences = (
        df_silver_clean.loc[valid_rows, column]
        - calculated_values.loc[valid_rows]
    ).abs()

    formula_results.append({
        "Column": column,
        "Comparable_Records": int(valid_rows.sum()),
        "Mismatches": int((differences > 0.02).sum()),
        "Maximum_Difference": (
            round(differences.max(), 4)
            if not differences.empty else None
        )
    })

display(pd.DataFrame(formula_results))

,Column,Comparable_Records,Mismatches,Maximum_Difference
0,Gross_Amount,2905,0,0.000
1,Discount_Amount,2757,0,0.005
2,Net_Sales,2757,0,0.000
3,Profit,2757,0,0.000
4,Profit_Margin_Pct,2756,0,0.005


In [82]:
# Cell 70: Final missing-value profile

silver_missing_profile = pd.DataFrame({
    "Column": df_silver_clean.columns,
    "Data_Type": df_silver_clean.dtypes.astype(str).values,
    "Missing_Count": df_silver_clean.isna().sum().values
})

silver_missing_profile["Missing_Percentage"] = (
    silver_missing_profile["Missing_Count"]
    / len(df_silver_clean) * 100
).round(2)

silver_missing_profile = (
    silver_missing_profile
    .sort_values("Missing_Count", ascending=False)
    .reset_index(drop=True)
)

display(silver_missing_profile)

print("\nTotal Silver records:", len(df_silver_clean))
print("Total columns:", len(df_silver_clean.columns))
print(
    "Total missing cells:",
    int(df_silver_clean.isna().sum().sum())
)


,Column,Data_Type,Missing_Count,Missing_Percentage
0,Profit_Margin_Pct,float64,239,7.98
1,Profit,float64,238,7.95
2,Estimated_Cost,float64,238,7.95
3,Discount_Amount,float64,238,7.95
4,Net_Sales,float64,238,7.95
5,Customer_Rating,float64,216,7.21
6,Customer_Satisfaction,float64,149,4.97
7,Discount_Pct,float64,149,4.97
8,Customer_Loyalty_Points,float64,119,3.97
9,Shipping_Days,float64,107,3.57



Total Silver records: 2995
Total columns: 33
Total missing cells: 2186


In [83]:
# Cell 71: Identify columns requiring data-type conversion

remaining_columns = [
    "Order_Date",
    "Items_Viewed",
    "Previous_Orders",
    "Delivery_Rating"
]

for col in remaining_columns:
    print(f"\n{col}")
    print("Current dtype:", df_silver_clean[col].dtype)
    print("Sample values:")
    display(df_silver_clean[col].dropna().unique()[:10])


Order_Date
Current dtype: str
Sample values:


<ArrowStringArray>
['2025-06-07', '2025-03-18', '2025-07-13', '2025-09-24', '2025-06-09',
 '2025-02-10', '2025-01-16', '2025-05-05', '2025-01-09', '2025-05-24']
Length: 10, dtype: str


Items_Viewed
Current dtype: str
Sample values:


<ArrowStringArray>
['10', '9', '4', '8', '6', '11', '5', '7', '3', '14']
Length: 10, dtype: str


Previous_Orders
Current dtype: str
Sample values:


<ArrowStringArray>
['10', '3', '6', '4', '8', '1', '7', '2', '11', '9']
Length: 10, dtype: str


Delivery_Rating
Current dtype: str
Sample values:


<ArrowStringArray>
['5.0', '3.6', '3.7', '3.5', '4.6', '2.9', '3.1', '2.8', '4.4', '4.3']
Length: 10, dtype: str

In [84]:
# Cell 72: Validate remaining data types

# Convert Order_Date
parsed_dates = pd.to_datetime(
    df_silver_clean["Order_Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Convert numerical columns
conversion_columns = [
    "Items_Viewed",
    "Previous_Orders",
    "Delivery_Rating"
]

conversion_results = []

for col in conversion_columns:
    converted = pd.to_numeric(
        df_silver_clean[col],
        errors="coerce"
    )

    conversion_results.append({
        "Column": col,
        "Original_Missing": df_silver_clean[col].isna().sum(),
        "Invalid_Values": (
            df_silver_clean[col].notna() & converted.isna()
        ).sum(),
        "Converted_Missing": converted.isna().sum()
    })

display(pd.DataFrame(conversion_results))

print("Invalid dates:", (
    df_silver_clean["Order_Date"].notna() & parsed_dates.isna()
).sum())

,Column,Original_Missing,Invalid_Values,Converted_Missing
0,Items_Viewed,0,0,0
1,Previous_Orders,0,0,0
2,Delivery_Rating,0,0,0


Invalid dates: 0


In [85]:
# Cell 73: Apply validated data-type conversions

df_silver_clean["Order_Date"] = pd.to_datetime(
    df_silver_clean["Order_Date"],
    format="%Y-%m-%d",
    errors="raise"
)

df_silver_clean["Items_Viewed"] = pd.to_numeric(
    df_silver_clean["Items_Viewed"],
    errors="raise"
).astype("Int64")

df_silver_clean["Previous_Orders"] = pd.to_numeric(
    df_silver_clean["Previous_Orders"],
    errors="raise"
).astype("Int64")

df_silver_clean["Delivery_Rating"] = pd.to_numeric(
    df_silver_clean["Delivery_Rating"],
    errors="raise"
).astype("Float64")

print("Updated data types:")
display(df_silver_clean.dtypes)

print("\nDate range:")
print(df_silver_clean["Order_Date"].min(), "to",
      df_silver_clean["Order_Date"].max())

Updated data types:


Order_ID                              str
Customer_ID                           str
Order_Date                 datetime64[us]
Customer_Age                      float64
Customer_Segment                      str
City                               string
Region                                str
Category                           string
Product_Name                          str
Sales_Channel                         str
Payment_Method                     string
Device                                str
Quantity                            int64
Unit_Price                        float64
Discount_Pct                      float64
Shipping_Days                     float64
Customer_Rating                   float64
Delivery_Rating                   Float64
Return_Status                         str
Order_Status                          str
Marketing_Source                      str
Customer_Loyalty_Points           float64
Website_Session_Minutes           float64
Items_Viewed                      


Date range:
2025-01-01 00:00:00 to 2025-12-31 00:00:00


In [86]:
# Cell 74: Final Silver integrity validation

silver_validation = {
    "Total_Records": len(df_silver_clean),
    "Total_Columns": len(df_silver_clean.columns),
    "Duplicate_Order_IDs": df_silver_clean["Order_ID"].duplicated().sum(),
    "Missing_Order_IDs": df_silver_clean["Order_ID"].isna().sum(),
    "Missing_Order_Dates": df_silver_clean["Order_Date"].isna().sum(),
    "Missing_City": df_silver_clean["City"].isna().sum(),
    "Missing_Payment_Method": df_silver_clean["Payment_Method"].isna().sum(),
    "Financial_Exception_Records": len(df_financial_exceptions)
}

display(
    pd.DataFrame(
        silver_validation.items(),
        columns=["Validation", "Result"]
    )
)

print("\nFinal DataFrame shape:", df_silver_clean.shape)
print("Unique Order IDs:", df_silver_clean["Order_ID"].nunique())
print("Financial audit records:", len(df_financial_exceptions))

,Validation,Result
0,Total_Records,2995
1,Total_Columns,33
2,Duplicate_Order_IDs,0
3,Missing_Order_IDs,0
4,Missing_Order_Dates,0
5,Missing_City,0
6,Missing_Payment_Method,0
7,Financial_Exception_Records,9



Final DataFrame shape: (2995, 33)
Unique Order IDs: 2995
Financial audit records: 9


In [87]:
# Cell 75: Inspect existing databases and tables

from sqlalchemy import text

with engine.connect() as connection:

    databases = pd.read_sql(
        text("SHOW DATABASES"),
        connection
    )

    existing_tables = pd.read_sql(
        text("""
            SELECT TABLE_SCHEMA, TABLE_NAME
            FROM information_schema.TABLES
            WHERE TABLE_SCHEMA IN (
                'ecommerce_analytics',
                'bronze',
                'silver',
                'gold'
            )
            ORDER BY TABLE_SCHEMA, TABLE_NAME
        """),
        connection
    )

print("Available MySQL databases:")
display(databases)

print("\nExisting project tables:")
display(existing_tables)

Available MySQL databases:


,Database
0,adventure_works_training
1,analytics
2,bank_churn
3,bronze
4,calibo_practice_1
5,calibodb
6,db_practice_1
7,ecommerce_analytics
8,ecommerce_dataset
9,gold



Existing project tables:


,TABLE_SCHEMA,TABLE_NAME
0,bronze,ecommerce_raw
1,silver,ecommerce_deduplicated
2,silver,ecommerce_duplicate_exceptions


In [88]:
# Cell 76: Inspect existing Silver candidate table

with engine.connect() as connection:

    silver_schema = pd.read_sql(
        text("""
            SELECT
                COLUMN_NAME,
                DATA_TYPE,
                IS_NULLABLE,
                COLUMN_KEY
            FROM information_schema.COLUMNS
            WHERE TABLE_SCHEMA = 'silver'
              AND TABLE_NAME = 'ecommerce_deduplicated'
            ORDER BY ORDINAL_POSITION
        """),
        connection
    )

    silver_count = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM silver.ecommerce_deduplicated
        """)
    ).scalar()

print("Existing Silver candidate schema:")
display(silver_schema)

print("\nExisting candidate records:", silver_count)
print("Cleaned DataFrame records:", len(df_silver_clean))

Existing Silver candidate schema:


,COLUMN_NAME,DATA_TYPE,IS_NULLABLE,COLUMN_KEY
0,Order_ID,text,YES,
1,Customer_ID,text,YES,
2,Order_Date,text,YES,
3,Customer_Age,text,YES,
4,Customer_Segment,text,YES,
5,City,text,YES,
6,Region,text,YES,
7,Category,text,YES,
8,Product_Name,text,YES,
9,Sales_Channel,text,YES,



Existing candidate records: 2995
Cleaned DataFrame records: 2995


In [89]:
# Cell 77: Inspect data for MySQL schema design

numeric_columns = [
    "Customer_Age", "Quantity", "Unit_Price", "Discount_Pct",
    "Shipping_Days", "Customer_Rating", "Delivery_Rating",
    "Customer_Loyalty_Points", "Website_Session_Minutes",
    "Items_Viewed", "Previous_Orders", "Customer_Satisfaction",
    "Gross_Amount", "Discount_Amount", "Net_Sales",
    "Estimated_Cost", "Profit", "Profit_Margin_Pct"
]

numeric_profile = []

for col in numeric_columns:
    values = pd.to_numeric(df_silver_clean[col], errors="coerce").dropna()

    numeric_profile.append({
        "Column": col,
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Fractional_Values": int(
            ((values % 1).abs() > 0.000001).sum()
        ),
        "Missing": int(df_silver_clean[col].isna().sum())
    })

display(pd.DataFrame(numeric_profile))

text_columns = df_silver_clean.select_dtypes(
    include=["object", "string"]
).columns

text_profile = pd.DataFrame([
    {
        "Column": col,
        "Maximum_Length": int(
            df_silver_clean[col].dropna().astype(str).str.len().max()
        )
    }
    for col in text_columns
])

display(text_profile)

,Column,Minimum,Maximum,Fractional_Values,Missing
0,Customer_Age,0.00,88.00,0,0
1,Quantity,-2.00,40.00,0,0
2,Unit_Price,12.75,61182.50,2877,90
3,Discount_Pct,-10.00,120.00,2636,149
4,Shipping_Days,1.00,35.00,0,107
5,Customer_Rating,1.00,5.00,2371,216
6,Delivery_Rating,1.00,5.00,2501,0
7,Customer_Loyalty_Points,0.00,2970.00,0,119
8,Website_Session_Minutes,1.00,96.10,2621,75
9,Items_Viewed,1.00,18.00,0,0


,Column,Maximum_Length
0,Order_ID,9
1,Customer_ID,5
2,Customer_Segment,10
3,City,10
4,Region,7
5,Category,15
6,Product_Name,10
7,Sales_Channel,11
8,Payment_Method,11
9,Device,7


In [90]:
# Cell 78: Check age boundary and numeric precision

print("Customer age boundary analysis:")

display(
    df_silver_clean["Customer_Age"]
    .value_counts()
    .sort_index()
    .head(10)
    .to_frame("Record_Count")
)

print("\nFractional precision of important columns:")

precision_columns = [
    "Unit_Price",
    "Discount_Pct",
    "Website_Session_Minutes",
    "Gross_Amount",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

display(
    pd.DataFrame({
        "Column": precision_columns,
        "Decimal_Places": [
            df_silver_clean[col].dropna()
            .map(lambda x: len(str(x).split(".")[-1]) if "." in str(x) else 0)
            .max()
            for col in precision_columns
        ]
    })
)

Customer age boundary analysis:


,Record_Count
Customer_Age,
0.0,1
1.0,3
4.0,1
5.0,8
6.0,4
7.0,3
8.0,2
9.0,6
10.0,4



Fractional precision of important columns:


,Column,Decimal_Places
0,Unit_Price,12
1,Discount_Pct,2
2,Website_Session_Minutes,1
3,Gross_Amount,2
4,Discount_Amount,2
5,Net_Sales,2
6,Estimated_Cost,2
7,Profit,2
8,Profit_Margin_Pct,2


In [91]:
# Cell 79: Inspect boundary value and floating-point precision

print("Record with Customer_Age = 0:")

display(
    df_silver_clean.loc[
        df_silver_clean["Customer_Age"] == 0
    ]
)

print("\nUnit_Price precision inspection:")

display(
    df_silver_clean["Unit_Price"]
    .dropna()
    .head(15)
    .to_frame("Original_Value")
    .assign(
        Rounded_Value=lambda x: x["Original_Value"].round(2),
        Difference=lambda x: (
            x["Original_Value"] - x["Original_Value"].round(2)
        )
    )
)

Record with Customer_Age = 0:


,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Items_Viewed,Previous_Orders,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct,Shipping_Data_Status
462,ORD101704,C1683,2025-03-14,0.0,Occasional,Chennai,South,Electronics,Headphones,Online,...,11,2,3.3,6916.53,1077.6,5838.93,4426.57,1412.36,24.19,Available



Unit_Price precision inspection:


,Original_Value,Rounded_Value,Difference
0,361.67,361.67,0.0
2,739.05,739.05,0.0
3,7001.11,7001.11,0.0
4,4920.89,4920.89,0.0
5,704.06,704.06,0.0
6,339.37,339.37,0.0
7,533.65,533.65,0.0
8,1052.89,1052.89,0.0
9,476.44,476.44,0.0
10,537.20,537.20,0.0


In [92]:
# Cell 80: Create final cleaned Silver table

create_silver_table = """
CREATE TABLE IF NOT EXISTS silver.ecommerce_cleaned (

    Order_ID VARCHAR(20) NOT NULL PRIMARY KEY,
    Customer_ID VARCHAR(20),
    Order_Date DATE,
    Customer_Age INT,
    Customer_Segment VARCHAR(50),
    City VARCHAR(50),
    Region VARCHAR(50),
    Category VARCHAR(50),
    Product_Name VARCHAR(100),
    Sales_Channel VARCHAR(50),
    Payment_Method VARCHAR(50),
    Device VARCHAR(50),

    Quantity INT,
    Unit_Price DECIMAL(12,2),
    Discount_Pct DECIMAL(7,2),
    Shipping_Days INT,
    Customer_Rating DECIMAL(3,1),
    Delivery_Rating DECIMAL(3,1),

    Return_Status VARCHAR(20),
    Order_Status VARCHAR(30),
    Marketing_Source VARCHAR(50),

    Customer_Loyalty_Points INT,
    Website_Session_Minutes DECIMAL(7,2),
    Items_Viewed INT,
    Previous_Orders INT,
    Customer_Satisfaction DECIMAL(3,1),

    Gross_Amount DECIMAL(12,2),
    Discount_Amount DECIMAL(12,2),
    Net_Sales DECIMAL(12,2),
    Estimated_Cost DECIMAL(12,2),
    Profit DECIMAL(12,2),
    Profit_Margin_Pct DECIMAL(7,2),

    Shipping_Data_Status VARCHAR(20)

)
"""

with engine.begin() as connection:
    connection.execute(text(create_silver_table))

print("Silver cleaned table created or already exists.")

Silver cleaned table created or already exists.


In [93]:
# Cell 81: Verify final Silver table structure

with engine.connect() as connection:

    silver_schema = pd.read_sql(
        text("""
            SELECT
                COLUMN_NAME,
                DATA_TYPE,
                COLUMN_TYPE,
                IS_NULLABLE,
                COLUMN_KEY
            FROM information_schema.COLUMNS
            WHERE TABLE_SCHEMA = 'silver'
              AND TABLE_NAME = 'ecommerce_cleaned'
            ORDER BY ORDINAL_POSITION
        """),
        connection
    )

    table_count = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM silver.ecommerce_cleaned
        """)
    ).scalar()

display(silver_schema)

print("\nTotal columns:", len(silver_schema))
print("Existing records:", table_count)

,COLUMN_NAME,DATA_TYPE,COLUMN_TYPE,IS_NULLABLE,COLUMN_KEY
0,Order_ID,varchar,varchar(20),NO,PRI
1,Customer_ID,varchar,varchar(20),YES,
2,Order_Date,date,date,YES,
3,Customer_Age,int,int,YES,
4,Customer_Segment,varchar,varchar(50),YES,
5,City,varchar,varchar(50),YES,
6,Region,varchar,varchar(50),YES,
7,Category,varchar,varchar(50),YES,
8,Product_Name,varchar,varchar(100),YES,
9,Sales_Channel,varchar,varchar(50),YES,



Total columns: 33
Existing records: 0


In [94]:
# Cell 82: Load validated Silver DataFrame into MySQL

# Prepare a separate DataFrame for database insertion
df_silver_load = df_silver_clean.copy()

# Columns requiring integer storage
integer_columns = [
    "Customer_Age",
    "Quantity",
    "Shipping_Days",
    "Customer_Loyalty_Points",
    "Items_Viewed",
    "Previous_Orders"
]

for col in integer_columns:
    df_silver_load[col] = (
        pd.to_numeric(df_silver_load[col], errors="raise")
        .astype("Int64")
        .astype(object)
        .where(df_silver_load[col].notna(), None)
    )

# Round DECIMAL columns to two places
decimal_columns = [
    "Unit_Price", "Discount_Pct", "Customer_Rating",
    "Delivery_Rating", "Website_Session_Minutes",
    "Customer_Satisfaction", "Gross_Amount",
    "Discount_Amount", "Net_Sales", "Estimated_Cost",
    "Profit", "Profit_Margin_Pct"
]

for col in decimal_columns:
    df_silver_load[col] = df_silver_load[col].round(2)

# Convert string columns to regular Python objects
string_columns = df_silver_load.select_dtypes(
    include=["object", "string"]
).columns

for col in string_columns:
    df_silver_load[col] = (
        df_silver_load[col]
        .astype(object)
        .where(df_silver_load[col].notna(), None)
    )

# Confirm destination is empty before insertion
with engine.begin() as connection:

    existing_count = connection.execute(
        text("SELECT COUNT(*) FROM silver.ecommerce_cleaned")
    ).scalar()

    if existing_count != 0:
        raise ValueError(
            "Destination table is not empty. Loading stopped."
        )

    df_silver_load.to_sql(
        name="ecommerce_cleaned",
        con=connection,
        schema="silver",
        if_exists="append",
        index=False,
        chunksize=500,
        method="multi"
    )

print("Silver data loaded successfully!")
print("Records submitted:", len(df_silver_load))

Silver data loaded successfully!
Records submitted: 2995


In [95]:
# Cell 83: Validate inserted Silver records

with engine.connect() as connection:

    # Record count
    mysql_count = connection.execute(
        text("SELECT COUNT(*) FROM silver.ecommerce_cleaned")
    ).scalar()

    # Unique Order IDs
    unique_orders = connection.execute(
        text("""
            SELECT COUNT(DISTINCT Order_ID)
            FROM silver.ecommerce_cleaned
        """)
    ).scalar()

    # Duplicate Order IDs
    duplicate_orders = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM (
                SELECT Order_ID
                FROM silver.ecommerce_cleaned
                GROUP BY Order_ID
                HAVING COUNT(*) > 1
            ) AS duplicates
        """)
    ).scalar()

    # Missing shipping values
    missing_shipping = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM silver.ecommerce_cleaned
            WHERE Shipping_Days IS NULL
        """)
    ).scalar()

    # Financial exception count
    financial_exceptions = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM silver.ecommerce_cleaned
            WHERE Quantity < 0
               OR Discount_Pct < 0
               OR Discount_Pct > 100
        """)
    ).scalar()

print("MYSQL SILVER VALIDATION")
print("-----------------------")
print("Python records:", len(df_silver_clean))
print("MySQL records:", mysql_count)
print("Unique Order IDs:", unique_orders)
print("Duplicate Order IDs:", duplicate_orders)
print("Missing Shipping Days:", missing_shipping)
print("Financial Exception Records:", financial_exceptions)

print("\nRecord count matches:",
      len(df_silver_clean) == mysql_count)

MYSQL SILVER VALIDATION
-----------------------
Python records: 2995
MySQL records: 2995
Unique Order IDs: 2995
Duplicate Order IDs: 0
Missing Shipping Days: 107
Financial Exception Records: 8

Record count matches: True


In [96]:
# Cell 84: Check financial exception audit table

with engine.connect() as connection:

    audit_table_exists = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM information_schema.TABLES
            WHERE TABLE_SCHEMA = 'silver'
              AND TABLE_NAME = 'ecommerce_financial_exceptions'
        """)
    ).scalar()

print("Financial exception table exists:", bool(audit_table_exists))

Financial exception table exists: False


In [97]:
# Cell 85: Export final cleaned Silver dataset

from pathlib import Path

project_root = Path.cwd()
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

silver_export_path = (
    processed_dir / "ecommerce_silver_cleaned.csv"
)

df_silver_clean.to_csv(
    silver_export_path,
    index=False
)

print("Silver dataset exported successfully!")
print("Records:", len(df_silver_clean))
print("Columns:", len(df_silver_clean.columns))
print("File:", silver_export_path.resolve())

Silver dataset exported successfully!
Records: 2995
Columns: 33
File: C:\Calibo_Course\Power BI\TRIVIKRAM_RAMESH_Ecommerce_Medallion_Project\data\processed\ecommerce_silver_cleaned.csv


In [98]:
# Cell 86: Export final Silver data-quality report

report_dir = Path.cwd() / "reports" / "data_quality"
report_dir.mkdir(parents=True, exist_ok=True)

silver_missing_profile.to_csv(
    report_dir / "silver_missing_value_profile.csv",
    index=False
)

print("Silver data-quality report exported.")
print("Missing-value profile:", 
      (report_dir / "silver_missing_value_profile.csv").exists())
print("Financial exception audit:",
      (report_dir / "financial_exception_audit.csv").exists())

Silver data-quality report exported.
Missing-value profile: True
Financial exception audit: True


In [99]:
# Cell 87: Final preprocessing export validation

# Read exported Silver dataset
df_exported = pd.read_csv(
    silver_export_path,
    parse_dates=["Order_Date"]
)

# Validate exported data
export_validation = {
    "Original_Records": len(df_silver_clean),
    "Exported_Records": len(df_exported),
    "Original_Columns": len(df_silver_clean.columns),
    "Exported_Columns": len(df_exported.columns),
    "Unique_Order_IDs": df_exported["Order_ID"].nunique(),
    "Duplicate_Order_IDs": df_exported["Order_ID"].duplicated().sum(),
    "Financial_Exception_Records": len(df_financial_exceptions),
    "Missing_Profile_Exists": (
        report_dir / "silver_missing_value_profile.csv"
    ).exists(),
    "Financial_Audit_Exists": (
        report_dir / "financial_exception_audit.csv"
    ).exists()
}

display(
    pd.DataFrame(
        export_validation.items(),
        columns=["Validation", "Result"]
    )
)

print("\nExport validation completed.")

,Validation,Result
0,Original_Records,2995
1,Exported_Records,2995
2,Original_Columns,33
3,Exported_Columns,33
4,Unique_Order_IDs,2995
5,Duplicate_Order_IDs,0
6,Financial_Exception_Records,9
7,Missing_Profile_Exists,True
8,Financial_Audit_Exists,True



Export validation completed.
